# DATA5925 — Data Preprocessing Pipeline

---

## Pipeline Overview

```
Stage 0   Global definitions (imports, constants, physiological bounds)
Stage 1   Parse raw patient files  (4,000 × {RecordID}.txt)
Stage 2   Stratified train / val / test split  (3200 / 400 / 400)
Stage 3   Shared temporal preprocessing
            │   (bounds filter → missingness mask → delta → pre-imputation copy)
            │
            ├── Stage 4   Flat benchmark branch
            │             → features_flat_raw_{split}.npy  (N, 306)  — tree models
            │             → features_flat_{split}.npy      (N, 306)  — linear / MLP
            │
            ├── Stage 5   Sequence benchmark branch
            │             → X_ts_{split}.npy      (N, 48, 37)  — LSTM / GRU / Transformer
            │             → X_static_{split}.npy  (N, 10)      — all sequence models
            │             → mask_{split}.npy       (N, 48, 37)  — GRU-D / BiT-MAC
            │             → delta_{split}.npy      (N, 48, 37)  — GRU-D / BiT-MAC
            │
            └── Stage 6   Irregular benchmark branch
                          → ts_irregular_{split}.parquet  — BiT-MAC (irregular variant)

Stage 7   Sensitivity analysis  (hard physiological bounds vs µ ± 5σ)
Stage 8   Validation checks     (21 assertions — run before any model training)
Stage 9   Save all outputs + metadata.json
Appendix  Quick-reference loading code
```

## Two-Track Design Rationale

Preprocessing forks into **Track A (flat)** and **Track B (sequence)** after Stage 3.
Neither track's choices affect the other:

| Track | Imputation | Log transform | Normalisation | Consumer models |
|---|---|---|---|---|
| **A — Flat** | None (observed values only) | log1p on observed positions | z-score per column | XGBoost, RF, LightGBM, LR, SVM, MLP |
| **B — Sequence** | LOCF → log1p → mean fill | log1p before fill (log-scale mean) | z-score per variable | LSTM, GRU, Transformer, GRU-D, BiT-MAC |

> **Run all cells top-to-bottom in the `temp_project` conda environment.**
> All outputs land in `processed/` (created automatically).
> Estimated runtime: 3–5 minutes on a modern laptop.

---

## Stage 0 — Global Definitions

Define file paths, global constants, and derived helpers used by every downstream stage.
Edit the `CONFIGURATION` block at the top of the next cell if your paths differ.

In [1]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*empty slice.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*All-NaN slice.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*Degrees of freedom.*')

# ---------------------------------------------------------------------------
# CONFIGURATION — edit these paths to match your local setup if needed
# ---------------------------------------------------------------------------
PATIENT_DIR   = Path('Patients')      # folder containing {RecordID}.txt files
OUTCOMES_FILE = Path('Outcomes-6.txt')
OUTPUT_DIR    = Path('processed')
# ---------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
VAL_RATIO   = 0.10   # 10% validation
TEST_RATIO  = 0.10   # 10% test  →  80% train
N_HOURS     = 48     # observation window: 48 hours post-ICU admission

# Variables that are static (recorded once at admission, not in the time-series grid)
STATIC_COLS       = ['Age', 'Gender', 'Height', 'ICUType']
# Variables that appear in BOTH the static branch (admission snapshot) and time-series branch
ADMISSION_EXTRAS  = ['Weight']
# Final static column layout after one-hot encoding ICUType (includes missingness indicators)
STATIC_FINAL_COLS = ['Age', 'Gender', 'Height', 'Weight',
                     'Height_missing', 'Weight_missing',
                     'ICUType_1', 'ICUType_2', 'ICUType_3', 'ICUType_4']
# Binary time-series variables: keep in {0,1}, do not z-score, impute missing with 0
TS_BINARY_COLS    = ['MechVent']

np.random.seed(RANDOM_SEED)
print(f'Patient directory : {PATIENT_DIR.resolve()}')
print(f'Outcomes file     : {OUTCOMES_FILE.resolve()}')
print(f'Output directory  : {OUTPUT_DIR.resolve()}')

# Right-skewed lab variables that receive log1p pre-transform before feature
# extraction and z-scoring.  Defined here once; used in both Track A (flat
# benchmark, observed-only log1p) and Track B (full-sequence log1p in Step 7S).
LOG_TRANSFORM_COLS = {
    'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Creatinine',
    'Glucose', 'Lactate', 'TroponinI', 'TroponinT', 'Urine', 'WBC'
}

Patient directory : D:\Coding\py\py_Project\DATA5925\Patients
Outcomes file     : D:\Coding\py\py_Project\DATA5925\Outcomes-6.txt
Output directory  : D:\Coding\py\py_Project\DATA5925\processed


### Physiological Reference Bounds

Hard bounds for all 37 time-series variables, derived from clinical literature and
the PhysioNet/CinC 2012 challenge guidelines.  Any recorded value outside these
ranges is treated as a transcription/sensor error and set to NaN before any
statistics or imputation are computed.

**Design choice — hard bounds vs. µ ± kσ:**
ICU lab distributions are heavily right-skewed and bimodal (survivors vs non-survivors),
so a fixed sigma cutoff is unreliable: it either retains physiologically impossible
values or discards genuine extreme cases.  Hard bounds encode domain knowledge
directly.  The σ-based alternative (Monteiro et al. 2020) is evaluated in Stage 7.

In [2]:
PHYS_BOUNDS = {
    # Liver enzymes
    'ALP':         (1,     5_000),   # Alkaline phosphatase  IU/L
    'ALT':         (1,    10_000),   # Alanine transaminase  IU/L
    'AST':         (1,    10_000),   # Aspartate transaminase IU/L
    'Albumin':     (0.5,     6.0),   # g/dL
    'Bilirubin':   (0.1,    80.0),   # mg/dL
    # Renal
    'BUN':         (1,      300),    # Blood urea nitrogen mg/dL
    'Cholesterol': (50,     600),    # mg/dL
    'Creatinine':  (0.1,    30.0),   # mg/dL
    # Blood pressure — invasive
    'DiasABP':     (1,      200),    # mmHg
    'MAP':         (10,     200),    # mmHg
    'SysABP':      (30,     300),    # mmHg
    # Blood pressure — non-invasive
    'NIDiasABP':   (1,      200),
    'NIMAP':       (10,     200),
    'NISysABP':    (30,     300),
    # Respiratory (bounds applied AFTER FiO2 unit conversion)
    'FiO2':        (0.21,   1.0),    # fraction; percentage values converted first
    'PaCO2':       (5,      200),    # mmHg
    'PaO2':        (20,     700),    # mmHg
    'RespRate':    (1,       80),    # bpm
    'SaO2':        (0,      100),    # %
    # Cardiac
    'HR':          (10,     300),    # bpm
    # Blood chemistry
    'Glucose':     (20,   1_000),    # mg/dL
    'HCO3':        (5,       60),    # mmol/L
    'HCT':         (5,       70),    # %
    'K':           (1,       12),    # mEq/L
    'Lactate':     (0.1,    30),     # mmol/L
    'Mg':          (0.5,     6.0),   # mmol/L
    'Na':          (100,    180),    # mEq/L
    'Platelets':   (5,    2_000),    # cells/nL
    'WBC':         (0.1,   200),     # cells/nL
    'pH':          (6.5,    8.0),
    # Cardiac biomarkers
    'TroponinI':   (0,    1_000),    # µg/L
    'TroponinT':   (0,      100),    # µg/L
    # Neurological
    'GCS':         (3,       15),
    # Other
    'MechVent':    (0,        1),    # binary: 0=not ventilated, 1=ventilated
    'Temp':        (25,      45),    # °C
    'Urine':       (0,    6_000),    # mL per hour
    'Weight':      (30,     250),    # kg (time-series version)
}

print(f'Physiological bounds defined for {len(PHYS_BOUNDS)} variables.')

Physiological bounds defined for 37 variables.


---

## Stage 1 — Parse Raw Patient Files

**What:** Read each `Data/Patients/{RecordID}.txt` file.  Each file contains
`Time,Parameter,Value` rows for one ICU stay (48 h post-admission).

**Key parsing decisions:**
- `Height` and `Weight` sentinel values (`-1.0`) → converted to `NaN` in `extract_static()`
- `build_hourly_grid()` aggregates to a regular 48-step hourly grid using `last()` per hour
  (last valid observation in each 60-minute window). Trade-off: simplicity and alignment
  with the PhysioNet challenge protocol; fine temporal detail within each hour is lost.
- `df_ts_irregular` keeps **all 37 time-series variables** at minute-level resolution,
  including `Weight` — matching the hourly branch so that the two BiT-MAC variants differ
  only in temporal resolution, not variable set.

**Outputs (in memory after this stage):**

| Variable | Shape / Size | Description |
|---|---|---|
| `df_static` | (4000, 6) | RecordID + Age, Gender, Height, ICUType, Weight |
| `df_ts` | (4000×48, 39) | Hourly grid per patient: RecordID, hour, 37 ts vars |
| `df_ts_irregular` | ~1.2 M rows | Raw long format: RecordID, minute, Parameter, Value |

In [3]:
def read_patient_long(path: Path) -> pd.DataFrame:
    """
    Read one patient .txt file into long format.
    Returns DataFrame with columns: Time, Parameter, Value, minute, hour, RecordID.
    Observations at or after the 48-hour boundary are discarded.
    """
    df = pd.read_csv(path, header=0)
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    t = pd.to_timedelta(df['Time'] + ':00')           # '01:20' -> timedelta
    df['minute']   = (t.dt.total_seconds() // 60).astype(int)
    df = df[df['minute'] < N_HOURS * 60].copy()       # discard anything >= 48h
    df['hour']     = (df['minute'] // 60).astype(int)
    df['RecordID'] = int(path.stem)
    return df


def extract_static(df_long: pd.DataFrame) -> dict:
    """
    Return one value per static variable (first valid observation).

    Sentinel-value handling:
      - Height and Weight recorded as -1 (or any value <= 0) are treated as missing.
      - Gender must be in {0, 1}; ICUType must be in {1, 2, 3, 4}.
      Any out-of-range static value is set to NaN and will be imputed downstream
      using training-set statistics (not a made-up constant).
    """
    POSITIVE_SENTINEL  = {'Height', 'Weight'}          # <= 0 means 'not recorded'
    VALID_BINARY       = {'Gender':  {0.0, 1.0}}
    VALID_CATEGORICAL  = {'ICUType': {1.0, 2.0, 3.0, 4.0}}

    result = {'RecordID': df_long['RecordID'].iloc[0]}

    for col in STATIC_COLS:
        vals = df_long.loc[df_long['Parameter'] == col, 'Value'].dropna()
        if col in POSITIVE_SENTINEL:
            vals = vals[vals > 0]
        if col in VALID_BINARY:
            vals = vals[vals.isin(VALID_BINARY[col])]
        if col in VALID_CATEGORICAL:
            vals = vals[vals.isin(VALID_CATEGORICAL[col])]
        result[col] = float(vals.iloc[0]) if len(vals) > 0 else np.nan

    # Admission-time Weight: use any observation in the first hour (hour == 0),
    # not just exact minute 00:00, since some records time-stamp it as e.g. 00:25.
    for col in ADMISSION_EXTRAS:
        admit = df_long.loc[
            (df_long['Parameter'] == col) & (df_long['hour'] == 0), 'Value'
        ].dropna()
        if col in POSITIVE_SENTINEL:
            admit = admit[admit > 0]
        result[col] = float(admit.iloc[0]) if len(admit) > 0 else np.nan

    return result


def build_hourly_grid(df_long: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse long format to a 48 × V wide DataFrame.

    Within each hour, the LAST valid observation is kept (clinically most current).
    Static variables (Age, Gender, Height, ICUType) are excluded — handled separately.
    Missing hours are present as NaN rows (ensures all patients have 48 rows).
    """
    record_id = df_long['RecordID'].iloc[0]
    EXCLUDE    = set(STATIC_COLS) | {'RecordID'}
    ts_df      = df_long[~df_long['Parameter'].isin(EXCLUDE)].copy()

    agg = (ts_df
           .sort_values(['hour', 'minute'])
           .groupby(['hour', 'Parameter'], as_index=False)['Value']
           .last())

    pivot = agg.pivot(index='hour', columns='Parameter', values='Value')
    pivot.columns.name = None
    pivot = pivot.reindex(range(N_HOURS))   # fill missing hours with NaN
    pivot.index.name = 'hour'
    pivot.insert(0, 'RecordID', record_id)
    return pivot.reset_index()

In [4]:
print('Loading patient files...')
static_list, ts_list, long_list = [], [], []

for fpath in sorted(PATIENT_DIR.glob('*.txt')):
    df_long = read_patient_long(fpath)
    static_list.append(extract_static(df_long))
    ts_list.append(build_hourly_grid(df_long))
    # Collect raw irregular long-format (actual timestamps, for BiT-MAC)
    _ts_long = df_long[~df_long['Parameter'].isin(set(STATIC_COLS) | {'RecordID'})]
    long_list.append(_ts_long[['RecordID', 'minute', 'Parameter', 'Value']].copy())

df_static = pd.DataFrame(static_list)              # [N_patients, 6]
df_ts           = pd.concat(ts_list,  ignore_index=True)  # [N_patients*48, 2+V]
df_ts_irregular = pd.concat(long_list, ignore_index=True)  # raw long-format, actual timestamps

all_record_ids = df_static['RecordID'].values
ts_cols        = sorted([c for c in df_ts.columns if c not in ['RecordID', 'hour']])
n_patients     = len(all_record_ids)

print(f'  Patients loaded    : {n_patients}')
print(f'  Time-series vars   : {len(ts_cols)}')
print(f'  Variables          : {ts_cols}')
print()

# --- Missingness overview (before any processing) ---
print('Missingness rate per variable (% of patient-hours with no observation):')
rates = {col: df_ts[col].isna().mean() * 100 for col in ts_cols}
for col, rate in sorted(rates.items(), key=lambda x: x[1], reverse=True):
    bar  = '#' * int(rate / 5)
    flag = '  <-- HIGH (>90%)' if rate > 90 else ''
    print(f'  {col:<15s}: {rate:5.1f}% {bar}{flag}')

Loading patient files...
  Patients loaded    : 4000
  Time-series vars   : 37
  Variables          : ['ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH']

Missingness rate per variable (% of patient-hours with no observation):
  Cholesterol    :  99.8% ###################  <-- HIGH (>90%)
  TroponinI      :  99.8% ###################  <-- HIGH (>90%)
  TroponinT      :  98.9% ###################  <-- HIGH (>90%)
  Albumin        :  98.8% ###################  <-- HIGH (>90%)
  ALP            :  98.4% ###################  <-- HIGH (>90%)
  ALT            :  98.3% ###################  <-- HIGH (>90%)
  AST            :  98.3% ###################  <-- HIGH (>90%)
  Bilirubin      :  98.3% ###########

---

## Stage 2 — Stratified Train / Val / Test Split

**What:** Load `Outcomes-6.txt` and partition 4,000 patients into
train (80%) / val (10%) / test (10%) using stratified sampling on `In-hospital_death`.

**Why stratified?**  Mortality rate is ~13.9%.  Without stratification, random chance
could produce a test set with much higher or lower mortality, making evaluation metrics
misleading.

> ⚠️ **Data leakage boundary.**  All normalisation parameters (means, standard deviations,
> imputation fallback values) must be fit on `train_ids` **only**.
> This split is the last step before any statistics touch the data.

**Outputs:**

| Variable | Size | Mortality rate |
|---|---|---|
| `train_ids`, `y_train` | 3200 | ~13.9% |
| `val_ids`, `y_val` | 400 | ~13.9% |
| `test_ids`, `y_test` | 400 | ~13.9% |

In [5]:
df_out = pd.read_csv(OUTCOMES_FILE)
labels = df_out.set_index('RecordID')['In-hospital_death']
y_all  = np.array([labels[rid] for rid in all_record_ids], dtype=np.int64)

print(f'Class distribution: {int(y_all.sum())} deaths / {len(y_all)} patients '
      f'({100 * y_all.mean():.1f}% mortality)')

# First split: separate out the 20% held-out (val + test)
train_ids, temp_ids, y_train, y_temp = train_test_split(
    all_record_ids, y_all,
    test_size   = VAL_RATIO + TEST_RATIO,
    stratify    = y_all,
    random_state= RANDOM_SEED,
)
# Second split: divide held-out equally into val and test
val_ids, test_ids, y_val, y_test = train_test_split(
    temp_ids, y_temp,
    test_size   = TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify    = y_temp,
    random_state= RANDOM_SEED,
)

print(f'\nSplit sizes  — train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}')
print(f'Mortality    — train: {100*y_train.mean():.1f}%  '
      f'val: {100*y_val.mean():.1f}%  test: {100*y_test.mean():.1f}%')
print('\nStratification successful: mortality rates are similar across all three splits.')

Class distribution: 554 deaths / 4000 patients (13.9% mortality)

Split sizes  — train: 3200 | val: 400 | test: 400
Mortality    — train: 13.8%  val: 14.0%  test: 13.8%

Stratification successful: mortality rates are similar across all three splits.


---

## Stage 3 — Shared Temporal Preprocessing

All three downstream branches (Flat / Sequence / Irregular) share this core.
**No imputation is applied here** — the shared layer delivers a bounds-filtered
3D array, a binary observation mask, per-variable time-since-last-observation deltas,
and a pre-imputation copy for the flat branch.

```
raw patient data (per split)
     │
     ├─ 3.1  Reshape to (N, 48, 37)  +  FiO2 unit fix  +  physiological bounds → NaN
     ├─ 3.2  Missingness mask          mask[n,t,v] = 1 if observed, 0 if missing
     │        └── pre-imputation copy  X_pre_locf  (Stage 4 reads this, never modified)
     └─ 3.3  Delta                     hours since last observation (from mask only)
```

**Why end the shared layer here?**
Placing imputation or log-scaling in the shared stage would force all branches to adopt
the same filling strategy.  Models that explicitly handle missingness (GRU-D, BiT-MAC)
need access to the raw missing pattern; models that aggregate statistics (XGBoost, RF)
should count only genuine observations.  Ending at the mask+delta keeps both options open.

### 3.1 — Reshape to 3D + Physiological Bounds Filter

Pivot each split's hourly table into a 3D numpy array of shape `(N, 48, 37)`, then
apply `PHYS_BOUNDS`: values outside the physiological range are set to `NaN`.

**FiO2 unit correction:** the PhysioNet 2012 dataset records FiO2 inconsistently —
some patients have fractional values (0.21–1.0), others have percentage values (21–100).
Values `> 1.0` are divided by 100 before the bounds check.

In [6]:
def df_to_3d(df: pd.DataFrame, record_ids: np.ndarray, cols: list) -> np.ndarray:
    """
    Vectorised conversion from the wide hourly DataFrame to a 3-D numpy array.
    Shape: [len(record_ids), N_HOURS, len(cols)].
    Row order matches the `record_ids` argument exactly.
    """
    full_idx = pd.MultiIndex.from_product(
        [record_ids, range(N_HOURS)], names=['RecordID', 'hour']
    )
    return (df
            .set_index(['RecordID', 'hour'])[cols]
            .reindex(full_idx)
            .values
            .reshape(len(record_ids), N_HOURS, len(cols))
            .astype(np.float32))


def apply_phys_bounds(X: np.ndarray, cols: list, verbose: bool = True) -> np.ndarray:
    """
    Set physiologically impossible values to NaN.
    Handles the FiO2 percentage-to-fraction conversion before bounds check.
    """
    X = X.copy()
    total_removed = 0
    for j, col in enumerate(cols):
        if col == 'FiO2':
            pct_mask    = X[:, :, j] > 1.0
            n_converted = int(pct_mask.sum())
            X[:, :, j]  = np.where(pct_mask, X[:, :, j] / 100.0, X[:, :, j])
            if n_converted > 0 and verbose:
                print(f'  FiO2: converted {n_converted} percentage value(s) to fraction (÷100)')

        if col in PHYS_BOUNDS:
            lo, hi     = PHYS_BOUNDS[col]
            bad        = (X[:, :, j] < lo) | (X[:, :, j] > hi)
            n          = int(bad.sum())
            if n > 0 and verbose:
                print(f'  {col:<15s}: removed {n:4d} value(s) outside [{lo}, {hi}]')
            X[:, :, j] = np.where(bad, np.nan, X[:, :, j])
            total_removed += n


    # Warn about any time-series variables without physiological bounds.
    # These variables are kept as-is — no outlier filtering is applied to them.
    # A missing entry is most likely a naming mismatch (e.g. TropI vs TroponinI).
    unmatched = [c for c in cols if c not in PHYS_BOUNDS and c != 'FiO2']
    if unmatched and verbose:
        print(f'  Warning — no bounds defined for: {unmatched}')
        print(f'    These variables are passed through unchanged.')

    if verbose:
        print(f'  Total removed: {total_removed} values')
    return X


# --- 3A: Reshape ---
X_train_raw = df_to_3d(df_ts, train_ids, ts_cols)
X_val_raw   = df_to_3d(df_ts, val_ids,   ts_cols)
X_test_raw  = df_to_3d(df_ts, test_ids,  ts_cols)
print(f'3D shapes — train: {X_train_raw.shape}  val: {X_val_raw.shape}  test: {X_test_raw.shape}')
print()

# --- 3B: Apply physiological bounds ---
print('Applying bounds (train):')
X_train = apply_phys_bounds(X_train_raw, ts_cols, verbose=True)
print('\nApplying bounds (val and test, silent):')
X_val   = apply_phys_bounds(X_val_raw,   ts_cols, verbose=False)
X_test  = apply_phys_bounds(X_test_raw,  ts_cols, verbose=False)
print('Bounds applied to val and test.')

3D shapes — train: (3200, 48, 37)  val: (400, 48, 37)  test: (400, 48, 37)

Applying bounds (train):
  ALT            : removed    5 value(s) outside [1, 10000]
  AST            : removed   17 value(s) outside [1, 10000]
  BUN            : removed    1 value(s) outside [1, 300]
  Cholesterol    : removed    2 value(s) outside [50, 600]
  DiasABP        : removed  348 value(s) outside [1, 200]
  Glucose        : removed    4 value(s) outside [20, 1000]
  HR             : removed    3 value(s) outside [10, 300]
  K              : removed    3 value(s) outside [1, 12]
  MAP            : removed  147 value(s) outside [10, 200]
  Mg             : removed    9 value(s) outside [0.5, 6.0]
  NIDiasABP      : removed   48 value(s) outside [1, 200]
  NIMAP          : removed   21 value(s) outside [10, 200]
  NISysABP       : removed   97 value(s) outside [30, 300]
  Na             : removed    1 value(s) outside [100, 180]
  PaCO2          : removed    1 value(s) outside [5, 200]
  PaO2         

### 3.2 — Missingness Mask + Pre-imputation Copy

```
mask[n, t, v] = 1   ← patient n had variable v measured at hour t
mask[n, t, v] = 0   ← no measurement recorded (NaN after bounds filter)
```

**Critical timing:** the mask is created here, BEFORE any imputation.
Once NaN is filled, "measured as zero" and "never measured" become indistinguishable.

**Pre-imputation copy** (`X_pre_locf`): Stage 4 (flat branch) reads from this copy
exclusively — it never sees LOCF-propagated or mean-filled values, ensuring flat
feature statistics reflect only genuine observations.

In [7]:
# Create masks: 1 = actually observed, 0 = NaN (missing)
# Must be done BEFORE imputation — once NaN is filled, we can no longer detect it.
mask_train = (~np.isnan(X_train)).astype(np.float32)   # shape: (N_train, 48, 37)
mask_val   = (~np.isnan(X_val  )).astype(np.float32)   # shape: (N_val,   48, 37)
mask_test  = (~np.isnan(X_test )).astype(np.float32)   # shape: (N_test,  48, 37)

print('Observation rate per variable (% of patient-hours with a real measurement):')
print(f'  {"Variable":<15s}  {"Train":>8s}  {"Val":>8s}  {"Test":>8s}')
print('  ' + '-' * 44)
for j, col in enumerate(ts_cols):
    r_tr = mask_train[:, :, j].mean() * 100
    r_va = mask_val  [:, :, j].mean() * 100
    r_te = mask_test [:, :, j].mean() * 100
    print(f'  {col:<15s}  {r_tr:7.1f}%  {r_va:7.1f}%  {r_te:7.1f}%')

# ─── Track A anchor: preserve pre-imputation arrays ───────────────────────
# Flat features (Track A) are extracted from observed values before any
# imputation.  Saving these references now makes the data flow explicit:
# flat feature extraction has zero dependence on the imputation strategy
# chosen for sequence models.
X_train_pre_locf = X_train.copy()
X_val_pre_locf   = X_val.copy()
X_test_pre_locf  = X_test.copy()
print()
print('Pre-imputation arrays preserved for Track A (flat benchmark).')

Observation rate per variable (% of patient-hours with a real measurement):
  Variable            Train       Val      Test
  --------------------------------------------
  ALP                  1.6%      1.7%      1.6%
  ALT                  1.6%      1.7%      1.7%
  AST                  1.6%      1.7%      1.7%
  Albumin              1.2%      1.3%      1.1%
  BUN                  7.2%      7.5%      7.3%
  Bilirubin            1.6%      1.7%      1.7%
  Cholesterol          0.2%      0.1%      0.2%
  Creatinine           7.2%      7.5%      7.3%
  DiasABP             54.1%     49.6%     53.7%
  FiO2                16.0%     16.0%     16.2%
  GCS                 31.8%     31.5%     32.4%
  Glucose              6.7%      7.1%      6.9%
  HCO3                 7.0%      7.4%      7.1%
  HCT                  9.5%      9.2%      9.4%
  HR                  89.8%     90.5%     91.1%
  K                    7.4%      7.8%      7.6%
  Lactate              4.0%      4.2%      4.4%
  MAP        

### 3.3 — Delta: Time Since Last Observation

```
delta[n, t, v] = t - last_obs_time    if variable v was observed at any t' < t
delta[n, t, v] = t                    if variable v has never been observed before hour t
```

Used by GRU-D and BiT-MAC as a temporal decay feature indicating how "stale" an
imputed or LOCF-filled value is.

**Computed from the mask only** (not from imputed values), so it correctly belongs
in the shared preprocessing stage rather than inside the sequence branch.

In [8]:
def compute_delta(mask_3d: np.ndarray) -> np.ndarray:
    """
    For each (patient, time-step, variable), compute hours since last observation.
    If the variable has never been observed before this time-step: delta = t (hours since admission).
    Shape: same as mask_3d, i.e. (N, T, V).
    """
    N, T, V  = mask_3d.shape
    delta    = np.zeros((N, T, V), dtype=np.float32)
    last_obs = np.full((N, V), -1.0, dtype=np.float32)   # -1 = never seen

    for t in range(T):
        observed   = mask_3d[:, t, :] == 1               # [N, V] bool
        # If never observed: delta = t  |  If previously observed: delta = t - last_obs_time
        delta[:, t, :] = np.where(last_obs >= 0, t - last_obs, float(t))
        last_obs       = np.where(observed, float(t), last_obs)

    return delta


delta_train = compute_delta(mask_train)
delta_val   = compute_delta(mask_val)
delta_test  = compute_delta(mask_test)

print(f'Delta computed.')
print(f'  Train — max: {delta_train.max():.0f}h, mean: {delta_train.mean():.2f}h')
print(f'  Val   — max: {delta_val.max():.0f}h,   mean: {delta_val.mean():.2f}h')
print(f'  Test  — max: {delta_test.max():.0f}h,  mean: {delta_test.mean():.2f}h')

Delta computed.
  Train — max: 47h, mean: 11.95h
  Val   — max: 47h,   mean: 11.92h
  Test  — max: 47h,  mean: 11.85h


---

## Stage 4 — Flat Benchmark Branch

**Consumer models:** XGBoost, Random Forest, LightGBM, Logistic Regression, SVM, MLP.

**Design principle:** extract summary statistics from *genuinely observed* positions
only (`mask == 1`, from `X_pre_locf`).  No imputation is applied before feature
extraction — flat models see only what was actually measured.

**Feature layout — 306 = 8 stats × 37 ts vars + 10 static:**

*Time-series block (296 cols):*
`{var}__{mean|std|min|max|first|last|count|missing_rate}` × 37 variables.
For right-skewed variables (`LOG_TRANSFORM_COLS`), stats are computed after applying
`log1p` to observed values.  Fallback for never-observed patients uses
`mean(log1p(x_obs))`, not `log1p(mean(x_obs))`.

*Static block (10 cols):*
`[Age, Gender, Height, Weight, Height_missing, Weight_missing, ICUType_1..4]`

**flat_raw vs flat_scaled:**

| Array | Scaling | Consumer |
|---|---|---|
| `features_flat_raw` | none (original units) | Tree models — scaling-invariant |
| `features_flat` | continuous cols z-scored; binary/indicator cols unchanged | Linear models / MLP |

`flat_scaled` is derived from `flat_raw` using a `continuous_mask` that only marks
ts mean/std/min/max/first/last and Age/Height/Weight for z-scoring.
Binary columns (count, missing_rate, Gender, Height_missing, Weight_missing, ICUType
one-hot) stay as-is, consistent with the methods description in the thesis.

### 4.1 — Static Variable Missingness Report

Report missing rates before any imputation.  These numbers should be cited in the
Methods section ("Variable X was missing in Y% of patients").

`Height` and `Weight` sentinel values (`-1`) were already converted to `NaN` in Stage 1.

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 9S pre-check: Static variable missingness
# Report before imputation so the team can include these numbers in the
# Methods section ("Variable X was missing for Y% of patients").
# Note: Height and Weight sentinel values (-1) were converted to NaN in Step 1.
# ─────────────────────────────────────────────────────────────────────────────

def get_static_df(df_static: pd.DataFrame, record_ids: np.ndarray) -> pd.DataFrame:
    return df_static.set_index('RecordID').reindex(record_ids).reset_index()

S_train_df = get_static_df(df_static, train_ids)
S_val_df   = get_static_df(df_static, val_ids)
S_test_df  = get_static_df(df_static, test_ids)

print('Static variable missingness before imputation:')
print(f'  {"Variable":<10s}  {"Train":>8s}  {"Val":>8s}  {"Test":>8s}')
print('  ' + '─' * 38)
for col in ['Age', 'Gender', 'Height', 'Weight', 'ICUType']:
    r_tr = S_train_df[col].isna().mean() * 100
    r_va = S_val_df[col].isna().mean() * 100
    r_te = S_test_df[col].isna().mean() * 100
    flag = '  ← HIGH' if r_tr > 20 else ''
    print(f'  {col:<10s}  {r_tr:7.1f}%  {r_va:7.1f}%  {r_te:7.1f}%{flag}')
print()
print('Missing values will be filled with training-set mean/mode in the next cell.')

Static variable missingness before imputation:
  Variable       Train       Val      Test
  ──────────────────────────────────────
  Age             0.0%      0.0%      0.0%
  Gender          0.1%      0.0%      0.2%
  Height         47.0%     50.7%     47.0%  ← HIGH
  Weight          7.4%      8.2%      7.0%
  ICUType         0.0%      0.0%      0.0%

Missing values will be filled with training-set mean/mode in the next cell.


### 4.2 — Static Feature Processing

Two static representations are produced, both with **10 columns**:

| Array | Col 0–3 encoding | Col 4–5 | Col 6–9 | Consumer |
|---|---|---|---|---|
| `S_norm` | Age, Height, Weight → z-scored; Gender → 0/1 | `Height_missing`, `Weight_missing` (0/1) | ICUType one-hot | Sequence models (Stage 5) |
| `S_raw_enc` | Age, Height, Weight → original units; Gender → 0/1 | same | same | flat_raw (tree models) |

**Missing indicators** are computed BEFORE `_fill_static()` runs — so the filled value
and the indicator flag are independent.  Both are kept as `{0, 1}` and are never
z-scored.

In [10]:
# --- Fit statistics on TRAINING SET ONLY ---
# S_train_df / S_val_df / S_test_df were created in the previous cell.
age_mean    = float(S_train_df['Age'].mean())
age_std     = float(max(S_train_df['Age'].std(), 1e-8))
height_mean = float(S_train_df['Height'].mean())
height_std  = float(max(S_train_df['Height'].std(), 1e-8))
weight_mean = float(S_train_df['Weight'].mean())
weight_std  = float(max(S_train_df['Weight'].std(), 1e-8))

_gv          = S_train_df['Gender'].dropna()
gender_mode  = float(_gv.mode().iloc[0]) if len(_gv) > 0 else 0.0
_iv          = S_train_df['ICUType'].dropna()
icutype_mode = int(_iv.mode().iloc[0]) if len(_iv) > 0 else 1

static_stats = {
    'age_mean': age_mean,       'age_std': age_std,
    'height_mean': height_mean, 'height_std': height_std,
    'weight_mean': weight_mean, 'weight_std': weight_std,
    'gender_mode': gender_mode, 'icutype_mode': icutype_mode,
}


def _fill_static(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing static values using train-set statistics (all splits)."""
    df = df.copy()
    df['Age']     = df['Age'].fillna(age_mean)
    df['Height']  = df['Height'].fillna(height_mean)
    df['Weight']  = df['Weight'].fillna(weight_mean)
    df['Gender']  = df['Gender'].fillna(gender_mode)
    df['ICUType'] = df['ICUType'].fillna(icutype_mode).astype(int)
    return df


def _icu_onehot(df: pd.DataFrame) -> np.ndarray:
    """One-hot encode ICUType {1,2,3,4} into 4 binary columns."""
    return (pd.get_dummies(df['ICUType'], prefix='ICUType')
              .reindex(columns=['ICUType_1', 'ICUType_2', 'ICUType_3', 'ICUType_4'],
                       fill_value=0)
              .values.astype(np.float32))


def process_static_norm(df: pd.DataFrame) -> np.ndarray:
    """
    Static features with continuous variables z-scored.
    Missing indicators are computed BEFORE imputation, then preserved as 0/1
    (they must not be z-scored).
    Used by: sequence models (Branches 9B, 9C).
    Layout: [Age_z, Gender, Height_z, Weight_z,
             Height_missing, Weight_missing,
             ICUType_1, ICUType_2, ICUType_3, ICUType_4]   shape (N, 10)
    """
    height_miss = df['Height'].isna().astype(np.float32).values
    weight_miss = df['Weight'].isna().astype(np.float32).values
    df       = _fill_static(df)
    age_z    = ((df['Age'].values    - age_mean)    / age_std   ).astype(np.float32)
    height_z = ((df['Height'].values - height_mean) / height_std).astype(np.float32)
    weight_z = ((df['Weight'].values - weight_mean) / weight_std).astype(np.float32)
    gender   =   df['Gender'].values.astype(np.float32)
    return np.column_stack([age_z, gender, height_z, weight_z,
                            height_miss, weight_miss, _icu_onehot(df)])


def process_static_raw_encoded(df: pd.DataFrame) -> np.ndarray:
    """
    Static features with continuous variables in ORIGINAL UNITS (not z-scored).
    Missing indicators are computed BEFORE imputation, then preserved as 0/1.
    Used by: flat_raw (Branch 9A) for tree models (XGBoost, RF, LightGBM).

    Tree models are invariant to monotone scaling, so z-scoring adds no predictive
    benefit.  Raw units also make SHAP values directly interpretable.

    Layout: [Age, Gender, Height, Weight,
             Height_missing, Weight_missing,
             ICUType_1, ICUType_2, ICUType_3, ICUType_4]   shape (N, 10)
    """
    height_miss = df['Height'].isna().astype(np.float32).values
    weight_miss = df['Weight'].isna().astype(np.float32).values
    df     = _fill_static(df)
    age    = df['Age'].values.astype(np.float32)
    height = df['Height'].values.astype(np.float32)
    weight = df['Weight'].values.astype(np.float32)
    gender = df['Gender'].values.astype(np.float32)
    return np.column_stack([age, gender, height, weight,
                            height_miss, weight_miss, _icu_onehot(df)])


# --- Produce both versions for all splits ---
S_train_norm    = process_static_norm(S_train_df)
S_val_norm      = process_static_norm(S_val_df)
S_test_norm     = process_static_norm(S_test_df)

S_train_raw_enc = process_static_raw_encoded(S_train_df)
S_val_raw_enc   = process_static_raw_encoded(S_val_df)
S_test_raw_enc  = process_static_raw_encoded(S_test_df)

print('Static features — two versions produced (now including missingness indicators)')
print(f'  S_norm (z-scored continuous + missing ind.):  '
      f'train {S_train_norm.shape}  val {S_val_norm.shape}  test {S_test_norm.shape}')
print(f'    → sequence models')
print(f'  S_raw_enc (original-unit + missing ind.):     '
      f'train {S_train_raw_enc.shape}  val {S_val_raw_enc.shape}  test {S_test_raw_enc.shape}')
print(f'    → flat_raw (tree models)')
print(f'  Layout: {STATIC_FINAL_COLS}')
h_miss_rate = S_train_df['Height'].isna().mean() * 100
w_miss_rate = S_train_df['Weight'].isna().mean() * 100
print(f'  Height_missing rate (train): {h_miss_rate:.1f}%')
print(f'  Weight_missing rate (train): {w_miss_rate:.1f}%')

Static features — two versions produced (now including missingness indicators)
  S_norm (z-scored continuous + missing ind.):  train (3200, 10)  val (400, 10)  test (400, 10)
    → sequence models
  S_raw_enc (original-unit + missing ind.):     train (3200, 10)  val (400, 10)  test (400, 10)
    → flat_raw (tree models)
  Layout: ['Age', 'Gender', 'Height', 'Weight', 'Height_missing', 'Weight_missing', 'ICUType_1', 'ICUType_2', 'ICUType_3', 'ICUType_4']
  Height_missing rate (train): 47.0%
  Weight_missing rate (train): 7.4%


### 4.3 — Time-Series Statistics + Flat Feature Assembly

### Feature Construction: 8 Statistics per Variable

For each of the 37 time-series variables, 8 summary statistics are computed
**only over mask == 1 positions** (genuine observations):

| Statistic | Clinical interpretation |
|---|---|
| `mean` | Average level of the biomarker over the observation window |
| `std` | Variability — high std in HR/BP may indicate haemodynamic instability |
| `min` | Worst-case low value (e.g. SaO2 nadir, lowest GCS) |
| `max` | Worst-case high value (e.g. fever peak, bilirubin peak) |
| `first` | Admission snapshot — baseline severity on arrival to ICU |
| `last` | Most recent measurement — end-of-window trajectory |
| `count` | Number of times the variable was measured (reflects monitoring intensity) |
| `missing_rate` | Fraction of hours with no measurement — signals rarely-measured variables |

**Why mask == 1 only?**
Computing statistics over the imputed array (including LOCF fill-ins) would bias
`mean` and `std` toward the last-observed value, conflating imputed constants with
real physiology. Using observed positions only ensures the statistics reflect actual
clinical measurements.

**Fallback for never-observed variables:**
For patients in whom a given variable was never measured, `mean/min/max/first/last`
are filled with the training-set mean of that variable (not 0.0, which would be
clinically nonsensical — e.g. HR = 0 implies cardiac arrest). `std` is set to 0.0.

**Log1p correction before stats:**
Right-skewed variables (liver enzymes, troponins, urine output, etc.) are
log1p-transformed on the observed positions before computing the statistics.
This ensures that `mean` and `std` are computed on an approximately Gaussian scale,
preventing extreme outliers from dominating.

**Total features:** 8 stats × 37 variables + 8 static = **304 features per patient**

In [ ]:
def extract_ts_stats(X_pre_locf, mask, fallback_means, log_indices):
    N, T, V = X_pre_locf.shape  
    feats = []
    for j in range(V):
        x = X_pre_locf[:, :, j]
        m = mask[:, :, j]
        x_obs = np.where(m == 1, x, np.nan).astype(np.float64)
        if j in log_indices:
            x_obs = np.where(m == 1, np.log1p(np.maximum(x_obs, 0.0)), np.nan)
        f_mean = np.nanmean(x_obs, axis=1)
        f_std  = np.nanstd(x_obs, axis=1)
        f_min  = np.nanmin(x_obs, axis=1)
        f_max  = np.nanmax(x_obs, axis=1)
        has_obs   = m.any(axis=1)
        first_idx = np.argmax(m, axis=1)
        last_idx  = T - 1 - np.argmax(m[:, ::-1], axis=1)
        row_indices = np.arange(N)  # local N
        first = np.where(has_obs, x_obs[row_indices, first_idx], np.nan)
        last  = np.where(has_obs, x_obs[row_indices, last_idx],  np.nan)
        f_count   = m.sum(axis=1).astype(np.float64)
        f_missing = 1.0 - m.mean(axis=1)
        never_obs = (f_count == 0)
        fallback  = float(fallback_means[j])
        f_mean[never_obs] = fallback
        f_min[never_obs]  = fallback
        f_max[never_obs]  = fallback
        first[never_obs]  = fallback
        last[never_obs]   = fallback
        f_std[never_obs]  = 0.0
        feats.extend([f_mean, f_std, f_min, f_max, first, last, f_count, f_missing])
    return np.column_stack(feats).astype(np.float32)


# Map LOG_TRANSFORM_COLS names → integer column indices in ts_cols.
log_col_indices = [j for j, col in enumerate(ts_cols) if col in LOG_TRANSFORM_COLS]

# Compute per-variable fallback means for 'never-observed' patients.
# Methodologically correct: compute mean(log1p(x)), NOT log1p(mean(x)).
# By Jensen's inequality for the concave log function: log(E[X]) >= E[log(X)],
# so log1p(mean) overestimates the mean of log-transformed values.
# The difference is small for most variables but is correct in principle.
flat_fallback_means = np.zeros(len(ts_cols), dtype=np.float32)
for j in range(len(ts_cols)):
    obs_vals = X_train_pre_locf[:, :, j][mask_train[:, :, j] == 1]
    if len(obs_vals) > 0:
        if j in log_col_indices:
            flat_fallback_means[j] = float(
                np.nanmean(np.log1p(np.maximum(obs_vals, 0.0)))
            )
        else:
            flat_fallback_means[j] = float(np.nanmean(obs_vals))
    # else: variable never observed in any train patient → stay at 0.0
flat_fallback_means = np.nan_to_num(flat_fallback_means, nan=0.0)


# Extract time-series statistics from pre-LOCF observed values
ts_stats_train = extract_ts_stats(X_train_pre_locf, mask_train, flat_fallback_means, log_col_indices)
ts_stats_val   = extract_ts_stats(X_val_pre_locf,   mask_val,   flat_fallback_means, log_col_indices)
ts_stats_test  = extract_ts_stats(X_test_pre_locf,  mask_test,  flat_fallback_means, log_col_indices)

# ─── flat_raw: raw ts_stats + static in original units ───────────────────────
# For XGBoost / RF / LightGBM — no scaling applied.
flat_raw_train = np.concatenate([ts_stats_train, S_train_raw_enc], axis=1).astype(np.float32)
flat_raw_val   = np.concatenate([ts_stats_val,   S_val_raw_enc  ], axis=1).astype(np.float32)
flat_raw_test  = np.concatenate([ts_stats_test,  S_test_raw_enc ], axis=1).astype(np.float32)

# ─── flat_scaled: derived from flat_raw, only continuous columns z-scored ─────
# For LR / SVM / MLP — fit scaler on training flat_raw only.
#
# Continuous columns (z-scored):  ts mean, std, min, max, first, last (6 per var)
#                                  + Age, Height, Weight in static block
# Non-continuous (kept as-is 0/1 or raw integer):
#                                  ts count, missing_rate (2 per var)
#                                  + Gender, Height_missing, Weight_missing,
#                                    ICUType_1..ICUType_4
#
# This fixes three issues from the prior version:
#   1. Age/Height/Weight were double z-scored (once in S_norm, once in flat_scaler)
#   2. Gender / ICUType one-hot were being scaled away from {0, 1}
#   3. New missing indicators (Height_missing, Weight_missing) must stay 0/1

n_ts_feats = 8 * len(ts_cols)                      # 296
n_flat     = n_ts_feats + len(STATIC_FINAL_COLS)   # 306

# Build boolean mask: True = column should be z-scored
# Flat layout: [ts_block (296)] + [Age(+0), Gender(+1), Height(+2), Weight(+3),
#               Height_miss(+4), Weight_miss(+5), ICUType_1..4(+6..+9)]
continuous_mask = np.zeros(n_flat, dtype=bool)
for _v in range(len(ts_cols)):
    _base = _v * 8
    continuous_mask[_base:_base+6] = True   # mean, std, min, max, first, last
    # count (+6) and missing_rate (+7): kept in natural units, not scaled
continuous_mask[n_ts_feats + 0] = True   # Age
continuous_mask[n_ts_feats + 2] = True   # Height
continuous_mask[n_ts_feats + 3] = True   # Weight
# Gender(+1), Height_missing(+4), Weight_missing(+5), ICUType(+6..+9): False

flat_mean = np.zeros(n_flat, dtype=np.float32)
flat_std  = np.ones( n_flat, dtype=np.float32)
flat_mean[continuous_mask] = flat_raw_train[:, continuous_mask].mean(axis=0).astype(np.float32)
flat_std [continuous_mask] = flat_raw_train[:, continuous_mask].std(axis=0).astype(np.float32)
flat_std  = np.where(flat_std == 0, 1.0, flat_std)

flat_scaled_train = ((flat_raw_train - flat_mean) / flat_std).astype(np.float32)
flat_scaled_val   = ((flat_raw_val   - flat_mean) / flat_std).astype(np.float32)
flat_scaled_test  = ((flat_raw_test  - flat_mean) / flat_std).astype(np.float32)

n_flat = flat_raw_train.shape[1]
print(f'Flat features assembled:')
print(f'  {len(ts_cols)} vars × 8 stats + {S_train_raw_enc.shape[1]} static = {n_flat} features per patient')
print()
print(f'  flat_raw    (unscaled — for trees):')
print(f'    train {flat_raw_train.shape}  val {flat_raw_val.shape}  test {flat_raw_test.shape}')
print(f'  flat_scaled (z-scored — for LR/SVM/MLP):')
print(f'    train {flat_scaled_train.shape}  val {flat_scaled_val.shape}  test {flat_scaled_test.shape}')

Flat features assembled:
  37 vars × 8 stats + 10 static = 306 features per patient

  flat_raw    (unscaled — for trees):
    train (3200, 306)  val (400, 306)  test (400, 306)
  flat_scaled (z-scored — for LR/SVM/MLP):
    train (3200, 306)  val (400, 306)  test (400, 306)


---

## Stage 5 — Sequence Benchmark Branch

**Consumer models:** LSTM, GRU, Transformer, GRU-D, BiT-MAC (hourly variant).

**Goal:** produce a fully imputed, normalised 3D array `(N, 48, 37)` where every
cell is a valid float.  Missingness-aware models (GRU-D, BiT-MAC) also use the
`mask` and `delta` arrays from Stage 3.

**Processing order — important, departs from naïve approach:**

```
Stage 3 output  (NaN where missing)
   → 5.1  LOCF              propagate last known value forward along time axis
   → 5.2  log1p             applied to LOG_TRANSFORM_COLS; NaN propagates safely
   → 5.2  mean fallback     fill remaining NaN with mean(log1p(x_obs))  ← log-scale mean
   → 5.3  z-score           fit μ, σ on train observed positions (mask == 1) only
```

**Why log1p before fill?**
If fill is done first with raw-scale means and then log1p is applied, the fallback
value becomes `log1p(mean_raw)`.  By Jensen's inequality (log is concave):
`log1p(E[X]) ≥ E[log1p(X)]`
so `log1p(mean_raw) > mean(log1p(x_obs))`.  Never-observed patients are systematically
assigned an inflated baseline.  Applying log1p first (NaN-safe) then filling with the
log-space mean avoids this — and is consistent with Stage 4's `flat_fallback_means`.

### 5.1 — LOCF: Last Observation Carried Forward

Forward-fill each variable along the time axis for every patient.

After LOCF, the only remaining `NaN` values are for variables that were **never
observed** for that patient (no prior value to carry forward).
Mean fallback is deliberately deferred to Step 5.2 so that fill values are in
the final (log-transformed) scale, not the raw scale.

In [12]:
def locf_3d(X: np.ndarray) -> np.ndarray:
    """Forward-fill NaN along the time axis (axis=1) for every patient/variable pair."""
    X = X.copy()
    for t in range(1, N_HOURS):
        missing    = np.isnan(X[:, t, :])
        X[:, t, :] = np.where(missing, X[:, t - 1, :], X[:, t, :])
    return X


def fill_with_means(X: np.ndarray, means: np.ndarray) -> np.ndarray:
    """Replace any remaining NaN with the pre-computed per-variable means."""
    X = X.copy()
    for j in range(X.shape[2]):
        nan_mask           = np.isnan(X[:, :, j])
        X[:, :, j][nan_mask] = means[j]
    return X


# --- Step A: LOCF ---
# Forward-fill to propagate the last known observation. After LOCF, the only
# remaining NaN values correspond to variables that were NEVER observed for a
# given patient (no prior value to propagate).
# Mean fallback is deliberately deferred to Step 7S (cell below), after log1p
# has been applied, so the fallback value is in the same scale as the final
# model input.  Using raw-scale means as fallback then log-transforming inflates
# the imputed values relative to the log-space mean — this ordering avoids that.
X_train = locf_3d(X_train)
X_val   = locf_3d(X_val)
X_test  = locf_3d(X_test)
print('Step A (LOCF) complete.')
print('Remaining NaN = variables that were never observed for that patient.')
print('Mean fallback will be applied after log1p transform (next cell).')

Step A (LOCF) complete.
Remaining NaN = variables that were never observed for that patient.
Mean fallback will be applied after log1p transform (next cell).


### 5.2 — Log1p Transform + Mean Fallback (Log Scale)

**Step B — log1p:** right-skewed variables receive `log1p(max(x, 0))`.
`NaN` propagates safely: `np.maximum(NaN, 0) = NaN`, `np.log1p(NaN) = NaN`.

**Step C — mean fallback:** `train_impute_means` is computed from `mask == 1`
positions in the **already log-transformed** array, giving `mean(log1p(x_obs))`.
Remaining `NaN` (never-observed variables) are then filled with these log-space means.

`train_impute_means[j]` for log variables should be interpreted as a **log-space mean**.
This is consistent with `flat_fallback_means` computed in Stage 4.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Track B — Sequence Adapter: Step 7S — Log1p + Mean fallback
#
# Correct ordering (fixes log/fill scale mismatch from earlier versions):
#   Step B: log1p transform on the LOCF-filled array.
#           NaN propagates through np.log1p(NaN) = NaN, so never-observed
#           positions remain NaN after this step.
#   Step C: Compute train_impute_means from ORIGINALLY OBSERVED positions
#           (mask==1) in the NOW log-transformed X_train.
#           → impute_mean = mean(log1p(x_obs)) [log-space mean]  ✓
#             NOT log1p(mean(x_obs))                              ✗ (old behaviour)
#           Then fill remaining NaN with those log-scale means.
# ─────────────────────────────────────────────────────────────────────────────

# Column indices for log transform (computed here, after ts_cols is available)
log_col_indices = [j for j, col in enumerate(ts_cols) if col in LOG_TRANSFORM_COLS]


def apply_log1p_transform(X: np.ndarray, col_indices: list) -> np.ndarray:
    """
    Apply log1p to the specified variable indices (returns a copy).
    np.maximum(..., 0.0) guards against negative values near the physiological
    lower bound — avoids NaN/-inf even if a bound is set to exactly 0.
    NaN values propagate unchanged (np.maximum(NaN, 0.0) = NaN in numpy).
    """
    X = X.copy()
    for j in col_indices:
        X[:, :, j] = np.log1p(np.maximum(X[:, :, j], 0.0))
    return X


# Step B: log1p (NaN-safe; remaining NaN = never-observed variables stay NaN)
X_train = apply_log1p_transform(X_train, log_col_indices)
X_val   = apply_log1p_transform(X_val,   log_col_indices)
X_test  = apply_log1p_transform(X_test,  log_col_indices)

print(f'Step B (log1p) applied to {len(log_col_indices)} variables (NaN preserved):')
for j in log_col_indices:
    print(f'  {ts_cols[j]}')

# Step C: Train-mean fallback in the log-transformed (final) scale.
# mask_train marks ORIGINALLY OBSERVED positions; at those positions, X_train
# now holds log1p(x_raw).  So np.nanmean(X_train[mask==1]) = mean(log1p(x_obs)),
# which is the correct log-space mean — consistent with flat_fallback_means in
# Branch 9A.
train_impute_means = np.array([
    float(np.nanmean(X_train[:, :, j][mask_train[:, :, j] == 1]))
    if mask_train[:, :, j].sum() > 0 else 0.0
    for j in range(len(ts_cols))
], dtype=np.float32)
train_impute_means = np.nan_to_num(train_impute_means, nan=0.0)

# Binary override: MechVent fallback = 0 (not ventilated is the clinical default).
for col in TS_BINARY_COLS:
    if col in ts_cols:
        train_impute_means[ts_cols.index(col)] = 0.0

# Apply fallback using TRAINING means only (no data leakage)
X_train = fill_with_means(X_train, train_impute_means)
X_val   = fill_with_means(X_val,   train_impute_means)
X_test  = fill_with_means(X_test,  train_impute_means)

# Force MechVent to 0 for any remaining NaN (safety net)
if 'MechVent' in ts_cols:
    j_mv = ts_cols.index('MechVent')
    for _X in [X_train, X_val, X_test]:
        _X[:, :, j_mv] = np.nan_to_num(_X[:, :, j_mv], nan=0.0)

assert not np.isnan(X_train).any(), 'NaN remaining in X_train after Step C!'
assert not np.isnan(X_val).any(),   'NaN remaining in X_val after Step C!'
assert not np.isnan(X_test).any(),  'NaN remaining in X_test after Step C!'
print('\nStep C (mean fallback + binary override) complete. No NaN remaining.')
print('train_impute_means scale check (log vars — values should be << raw_obs_mean):')
for j in log_col_indices[:5]:
    print(f'  {ts_cols[j]:<15s}: impute_mean = {train_impute_means[j]:.4f} (log-space)')

Step B (log1p) applied to 12 variables (NaN preserved):
  ALP
  ALT
  AST
  BUN
  Bilirubin
  Creatinine
  Glucose
  Lactate
  TroponinI
  TroponinT
  Urine
  WBC

Step C (mean fallback + binary override) complete. No NaN remaining.
train_impute_means scale check (log vars — values should be << raw_obs_mean):
  ALP            : impute_mean = 4.4981 (log-space)
  ALT            : impute_mean = 4.2276 (log-space)
  AST            : impute_mean = 4.5437 (log-space)
  BUN            : impute_mean = 3.0900 (log-space)
  Bilirubin      : impute_mean = 0.9071 (log-space)


### 5.3 — Z-score Normalisation

Standardise: `z = (x − μ) / σ` where μ, σ are fit on **training observed positions
only** (`mask_train == 1`).  LOCF-filled and mean-filled positions do not influence
the normalisation parameters — only genuine measurements do.

`MechVent` is excluded (binary variable; scaling would distort `{0, 1}`).

In [14]:
exclude_norm = set(TS_BINARY_COLS)   # MechVent stays in {0, 1}

norm_mean = np.zeros(len(ts_cols), dtype=np.float32)
norm_std  = np.ones( len(ts_cols), dtype=np.float32)

# Fit on TRAINING SET ONLY, using originally-observed positions (mask_train == 1)
for j, col in enumerate(ts_cols):
    if col in exclude_norm:
        continue
    observed = X_train[:, :, j][mask_train[:, :, j] == 1]
    if len(observed) > 1:
        norm_mean[j] = float(observed.mean())
        s            = float(observed.std())
        norm_std[j]  = s if s > 0 else 1.0


def zscore(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / std).astype(np.float32)


X_train_norm = zscore(X_train, norm_mean, norm_std)
X_val_norm   = zscore(X_val,   norm_mean, norm_std)
X_test_norm  = zscore(X_test,  norm_mean, norm_std)

# Restore binary variables (z-score must not change them)
for col in exclude_norm:
    if col in ts_cols:
        j = ts_cols.index(col)
        X_train_norm[:, :, j] = X_train[:, :, j]
        X_val_norm  [:, :, j] = X_val  [:, :, j]
        X_test_norm [:, :, j] = X_test [:, :, j]

print('Z-score normalisation complete (fit on training observed values only).')
print(f'  Example — first 5 variables (train, observed only):')
for j in range(min(5, len(ts_cols))):
    print(f'    {ts_cols[j]:<15s}: mean = {norm_mean[j]:+7.3f},  std = {norm_std[j]:.3f}')

Z-score normalisation complete (fit on training observed values only).
  Example — first 5 variables (train, observed only):
    ALP            : mean =  +4.498,  std = 0.631
    ALT            : mean =  +4.228,  std = 1.584
    AST            : mean =  +4.544,  std = 1.499
    Albumin        : mean =  +2.907,  std = 0.646
    BUN            : mean =  +3.090,  std = 0.681


---

## Stage 6 — Irregular Benchmark Branch

**Consumer models:** BiT-MAC (irregular variant) and any method that operates on
raw-timestamp observations without hourly aggregation.

**Variable alignment with Stage 5:**
Both hourly and irregular benchmarks use the **same 37 time-series variables**
(including `Weight`) and apply the same physiological bounds and FiO2 unit fix.
Any performance difference between the two BiT-MAC variants can therefore be
attributed solely to temporal resolution, not to a different feature set.

**Per-split export:**
Three separate parquet files (`ts_irregular_train/val/test.parquet`) prevent
downstream code from accidentally loading the full dataset and re-splitting,
which would cause data leakage.

**Output columns:** `RecordID` (int), `minute` (int), `Parameter` (str), `Value` (float32)

In [15]:
# ─────────────────────────────────────────────────────────────────────
# Step 3C — Save bounds-filtered irregular data for BiT-MAC (one file per split)
# Apply the same physiological bounds as 3B, but on the raw long format.
# Exporting per-split files prevents downstream code from accidentally
# re-splitting or mixing train/val/test observations.
# Output: processed/ts_irregular_{train|val|test}.parquet
#   Columns: RecordID (int), minute (int), Parameter (str), Value (float32)
# NOTE: requires pyarrow — install with: pip install pyarrow
# ─────────────────────────────────────────────────────────────────────
df_irr = df_ts_irregular.copy()
df_irr['Value'] = df_irr['Value'].astype('float32')

# FiO2 unit fix (identical to apply_phys_bounds)
fio2_pct = (df_irr['Parameter'] == 'FiO2') & (df_irr['Value'] > 1.0)
df_irr.loc[fio2_pct, 'Value'] = df_irr.loc[fio2_pct, 'Value'] / 100.0

# Apply per-variable bounds: flag out-of-range values as NaN then drop
for _param, (_lo, _hi) in PHYS_BOUNDS.items():
    _bad = (df_irr['Parameter'] == _param) & (
        (df_irr['Value'] < _lo) | (df_irr['Value'] > _hi)
    )
    df_irr.loc[_bad, 'Value'] = float('nan')
df_irr = df_irr.dropna(subset=['Value']).reset_index(drop=True)

# Export one file per split (train_ids / val_ids / test_ids from Step 2)
for split_name, split_ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    df_split = df_irr[df_irr['RecordID'].isin(split_ids)].copy()
    out_path = OUTPUT_DIR / f'ts_irregular_{split_name}.parquet'
    df_split.to_parquet(out_path, index=False)
    print(f'  {split_name:5s}: {len(df_split):>10,} obs  |  '
          f'{df_split["RecordID"].nunique():>4d} patients  →  {out_path.name}')

# Verify variable set matches hourly time-series variables
irr_vars = set(df_irr['Parameter'].unique())
if irr_vars == set(ts_cols):
    print(f'\n[OK] Irregular variable set matches hourly ts_cols ({len(irr_vars)} vars)')
else:
    print(f'\n[WARN] Irregular / hourly variable mismatch: '
          f'{irr_vars.symmetric_difference(set(ts_cols))}')

  train:  1,384,573 obs  |  3198 patients  →  ts_irregular_train.parquet
  val  :    172,889 obs  |   399 patients  →  ts_irregular_val.parquet
  test :    177,065 obs  |   400 patients  →  ts_irregular_test.parquet

[OK] Irregular variable set matches hourly ts_cols (37 vars)


---

## Stage 7 — Sensitivity Analysis: Hard Physiological Bounds vs µ ± 5σ

**Purpose:** quantify how the choice of outlier-removal strategy affects observation
retention and downstream inputs.

| Strategy | Source | Bounds definition |
|---|---|---|
| **Main pipeline** | Domain literature | Hard physiological limits (`PHYS_BOUNDS`) |
| **Alternative** | Monteiro et al. (2020), *J. Biomed. Inf.* | µ ± 5σ from healthy-population norms |

**Cell 1 (below):** comparison table — observations removed per variable under each strategy.

**Cell 2 (below):** full alternate pipeline using µ ± 5σ bounds, with all main-pipeline
fixes applied (correct log/fill order, flat_scaled from flat_raw, updated static shapes).
Output saved to `processed/sensitivity_ksigma/`.

**How to use:** swap `processed/` for `processed/sensitivity_ksigma/` in your model
training notebook and rerun; everything else stays identical.

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sensitivity Analysis: µ ± 5σ bounds (Monteiro et al., 2020)
# Reference values from USP paper Appendix A — healthy-population norms.
# ═══════════════════════════════════════════════════════════════════════════════

KSIGMA_DIR = OUTPUT_DIR / 'sensitivity_ksigma'
KSIGMA_DIR.mkdir(parents=True, exist_ok=True)

# USP reference µ and σ (from Appendix A / Table 1 of Monteiro et al.)
# Format: variable_name → (µ, σ)
USP_REF = {
    'ALP':        (64,     28),
    'ALT':        (20,     15),
    'AST':        (23.5,   16.5),
    'Albumin':    (4.25,   0.75),
    'Bilirubin':  (0.75,   0.55),
    'BUN':        (34,     12),
    'Cholesterol':(100,    100),
    'Creatinine': (0.9,    0.3),
    'DiasABP':    (70,     30),
    'FiO2':       (0.5,    0.5),
    'GCS':        (9,      6),
    'Glucose':    (90,     20),
    'HCO3':       (26,     3),
    'HCT':        (45.5,   8.5),
    'HR':         (73,     25),
    'K':          (4.3,    0.8),
    'Lactate':    (0.8,    0.5),
    'MAP':        (100,    45),
    'Mg':         (1.7,    0.4),
    'Na':         (140,    5),
    'NIDiasABP':  (70,     30),
    'NIMAP':      (100,    45),
    'NISysABP':   (130,    60),
    'PaCO2':      (40,     5),
    'PaO2':       (90,     10),
    'pH':         (7.4,    0.05),
    'Platelets':  (275,    125),
    'RespRate':   (18,     2),
    'SaO2':       (96.5,   2.5),
    'SysABP':     (130,    60),
    'Temp':       (36.65,  0.55),
    'TroponinI':  (0.02,   0.02),
    'TroponinT':  (0.05,   0.05),
    'Urine':      (246.79, 17.93),
    'WBC':        (7.75,   3.25),
    'Weight':     (115,    85),
    'MechVent':   (0.5,    0.5),    # binary — bounds will be [0, 1] effectively
}

K_SIGMA = 5

# Build µ±5σ bounds dict (clamp lower bound to 0 for variables that can't be negative)
NON_NEGATIVE_VARS = set(ts_cols) - {'Temp', 'pH'}  # only Temp and pH can go below 0 theoretically
KSIGMA_BOUNDS = {}
for var in ts_cols:
    if var in USP_REF:
        mu, sigma = USP_REF[var]
        lo = mu - K_SIGMA * sigma
        hi = mu + K_SIGMA * sigma
        if var in NON_NEGATIVE_VARS:
            lo = max(lo, 0.0)
        KSIGMA_BOUNDS[var] = (lo, hi)

# ─── Comparison table ─────────────────────────────────────────────────────────
print(f'{"Variable":<15s}  {"Hard lo":>8s} {"Hard hi":>8s}  │  '
      f'{"k5σ lo":>8s} {"k5σ hi":>8s}  │  '
      f'{"Hard rm":>8s} {"k5σ rm":>8s} {"Extra":>8s}')
print('─' * 95)

# Apply k-sigma bounds to the SAME raw arrays (post-FiO2-fix, pre-hard-bounds)
# We need the FiO2-corrected but otherwise unfiltered data.
# X_train_raw already had FiO2 converted in apply_phys_bounds — but that also
# applied hard bounds. So we re-derive from the original raw arrays.
X_train_fio2 = X_train_raw.copy()
X_val_fio2   = X_val_raw.copy()
X_test_fio2  = X_test_raw.copy()
for _arr in [X_train_fio2, X_val_fio2, X_test_fio2]:
    if 'FiO2' in ts_cols:
        j_fio2 = ts_cols.index('FiO2')
        pct = _arr[:, :, j_fio2] > 1.0
        _arr[:, :, j_fio2] = np.where(pct, _arr[:, :, j_fio2] / 100.0, _arr[:, :, j_fio2])

total_hard = 0
total_ksig = 0
rows = []
for j, col in enumerate(ts_cols):
    all_obs = np.concatenate([
        X_train_fio2[:, :, j].ravel(),
        X_val_fio2[:, :, j].ravel(),
        X_test_fio2[:, :, j].ravel()
    ])
    valid = all_obs[~np.isnan(all_obs)]

    h_lo, h_hi = PHYS_BOUNDS.get(col, (np.nan, np.nan))
    hard_rm = int(((valid < h_lo) | (valid > h_hi)).sum()) if col in PHYS_BOUNDS else 0

    if col in KSIGMA_BOUNDS:
        k_lo, k_hi = KSIGMA_BOUNDS[col]
        ksig_rm = int(((valid < k_lo) | (valid > k_hi)).sum())
    else:
        k_lo, k_hi = np.nan, np.nan
        ksig_rm = 0

    extra = ksig_rm - hard_rm
    total_hard += hard_rm
    total_ksig += ksig_rm

    flag = '  ◄ TIGHTER' if extra > 50 else ''
    print(f'{col:<15s}  {h_lo:8.1f} {h_hi:8.1f}  │  '
          f'{k_lo:8.2f} {k_hi:8.2f}  │  '
          f'{hard_rm:8d} {ksig_rm:8d} {extra:+8d}{flag}')
    rows.append({'variable': col, 'hard_lo': h_lo, 'hard_hi': h_hi,
                 'ksig_lo': k_lo, 'ksig_hi': k_hi,
                 'hard_removed': hard_rm, 'ksig_removed': ksig_rm,
                 'extra_removed': extra})

print('─' * 95)
print(f'{"TOTAL":<15s}  {"":>8s} {"":>8s}  │  '
      f'{"":>8s} {"":>8s}  │  '
      f'{total_hard:8d} {total_ksig:8d} {total_ksig - total_hard:+8d}')
print()
print(f'Hard bounds remove {total_hard} values total.')
print(f'µ±5σ removes {total_ksig} values total ({total_ksig - total_hard:+d} additional).')
print(f'The µ±5σ method discards {(total_ksig - total_hard) / max(total_hard, 1) * 100:.0f}% '
      f'more observations than hard bounds.')

pd.DataFrame(rows).to_csv(KSIGMA_DIR / 'bounds_comparison.csv', index=False)
print(f'\nComparison table saved → {KSIGMA_DIR / "bounds_comparison.csv"}')

Variable          Hard lo  Hard hi  │    k5σ lo   k5σ hi  │   Hard rm   k5σ rm    Extra
───────────────────────────────────────────────────────────────────────────────────────────────
ALP                   1.0   5000.0  │      0.00   204.00  │         0      314     +314  ◄ TIGHTER
ALT                   1.0  10000.0  │      0.00    95.00  │         8     1018    +1010  ◄ TIGHTER
AST                   1.0  10000.0  │      0.00   106.00  │        17     1146    +1129  ◄ TIGHTER
Albumin               0.5      6.0  │      0.50     8.00  │         0        0       +0
BUN                   1.0    300.0  │      0.00    94.00  │         1      354     +353  ◄ TIGHTER
Bilirubin             0.1     80.0  │      0.00     3.50  │         0      548     +548  ◄ TIGHTER
Cholesterol          50.0    600.0  │      0.00   600.00  │         2        0       -2
Creatinine            0.1     30.0  │      0.00     2.40  │         0     1811    +1811  ◄ TIGHTER
DiasABP               1.0    200.0  │      0.0

In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# Produce alternate processed files under k-sigma bounds
# Reuses every function from the main pipeline — only the bounds differ.
# Output directory: processed/sensitivity_ksigma/
# ═══════════════════════════════════════════════════════════════════════════════

def apply_ksigma_bounds(X: np.ndarray, cols: list) -> np.ndarray:
    """Apply µ±5σ bounds using USP clinical reference values."""
    X = X.copy()
    if 'FiO2' in cols:
        j = cols.index('FiO2')
        pct = X[:, :, j] > 1.0
        X[:, :, j] = np.where(pct, X[:, :, j] / 100.0, X[:, :, j])
    for j, col in enumerate(cols):
        if col in KSIGMA_BOUNDS:
            lo, hi = KSIGMA_BOUNDS[col]
            bad = (X[:, :, j] < lo) | (X[:, :, j] > hi)
            X[:, :, j] = np.where(bad, np.nan, X[:, :, j])
    return X


# --- Step 3: Apply k-sigma bounds ---
kX_train = apply_ksigma_bounds(X_train_raw, ts_cols)
kX_val   = apply_ksigma_bounds(X_val_raw,   ts_cols)
kX_test  = apply_ksigma_bounds(X_test_raw,  ts_cols)

# --- Step 4: Mask ---
kmask_train = (~np.isnan(kX_train)).astype(np.float32)
kmask_val   = (~np.isnan(kX_val  )).astype(np.float32)
kmask_test  = (~np.isnan(kX_test )).astype(np.float32)

kX_train_pre_locf = kX_train.copy()
kX_val_pre_locf   = kX_val.copy()
kX_test_pre_locf  = kX_test.copy()

# --- Step 5S: Delta ---
kdelta_train = compute_delta(kmask_train)
kdelta_val   = compute_delta(kmask_val)
kdelta_test  = compute_delta(kmask_test)

# --- Step 6S: LOCF ---
kX_train = locf_3d(kX_train)
kX_val   = locf_3d(kX_val)
kX_test  = locf_3d(kX_test)

# --- Step 7S: Log1p (before mean fill — mirrors main pipeline fix) ---
kX_train = apply_log1p_transform(kX_train, log_col_indices)
kX_val   = apply_log1p_transform(kX_val,   log_col_indices)
kX_test  = apply_log1p_transform(kX_test,  log_col_indices)

# --- Mean fallback in log-transformed scale ---
k_impute_means = np.array([
    float(np.nanmean(kX_train[:, :, j][kmask_train[:, :, j] == 1]))
    if kmask_train[:, :, j].sum() > 0 else 0.0
    for j in range(len(ts_cols))
], dtype=np.float32)
k_impute_means = np.nan_to_num(k_impute_means, nan=0.0)
for col in TS_BINARY_COLS:
    if col in ts_cols:
        k_impute_means[ts_cols.index(col)] = 0.0

kX_train = fill_with_means(kX_train, k_impute_means)
kX_val   = fill_with_means(kX_val,   k_impute_means)
kX_test  = fill_with_means(kX_test,  k_impute_means)
if 'MechVent' in ts_cols:
    j_mv = ts_cols.index('MechVent')
    for _a in [kX_train, kX_val, kX_test]:
        _a[:, :, j_mv] = np.nan_to_num(_a[:, :, j_mv], nan=0.0)

# --- Step 8S: Z-score (fit on k-sigma train observed) ---
k_norm_mean = np.zeros(len(ts_cols), dtype=np.float32)
k_norm_std  = np.ones( len(ts_cols), dtype=np.float32)
for j, col in enumerate(ts_cols):
    if col in exclude_norm:
        continue
    obs = kX_train[:, :, j][kmask_train[:, :, j] == 1]
    if len(obs) > 1:
        k_norm_mean[j] = float(obs.mean())
        s = float(obs.std())
        k_norm_std[j] = s if s > 0 else 1.0

kX_train_norm = zscore(kX_train, k_norm_mean, k_norm_std)
kX_val_norm   = zscore(kX_val,   k_norm_mean, k_norm_std)
kX_test_norm  = zscore(kX_test,  k_norm_mean, k_norm_std)
for col in exclude_norm:
    if col in ts_cols:
        j = ts_cols.index(col)
        kX_train_norm[:, :, j] = kX_train[:, :, j]
        kX_val_norm  [:, :, j] = kX_val  [:, :, j]
        kX_test_norm [:, :, j] = kX_test [:, :, j]

# --- Branch 9A: Flat features ---
k_flat_fallback = np.zeros(len(ts_cols), dtype=np.float32)
for j in range(len(ts_cols)):
    obs = kX_train_pre_locf[:, :, j][kmask_train[:, :, j] == 1]
    if len(obs) > 0:
        if j in log_col_indices:
            k_flat_fallback[j] = float(np.nanmean(np.log1p(np.maximum(obs, 0.0))))
        else:
            k_flat_fallback[j] = float(np.nanmean(obs))
k_flat_fallback = np.nan_to_num(k_flat_fallback, nan=0.0)

k_ts_stats_train = extract_ts_stats(kX_train_pre_locf, kmask_train, k_flat_fallback, log_col_indices)
k_ts_stats_val   = extract_ts_stats(kX_val_pre_locf,   kmask_val,   k_flat_fallback, log_col_indices)
k_ts_stats_test  = extract_ts_stats(kX_test_pre_locf,  kmask_test,  k_flat_fallback, log_col_indices)

k_flat_raw_train = np.concatenate([k_ts_stats_train, S_train_raw_enc], axis=1).astype(np.float32)
k_flat_raw_val   = np.concatenate([k_ts_stats_val,   S_val_raw_enc  ], axis=1).astype(np.float32)
k_flat_raw_test  = np.concatenate([k_ts_stats_test,  S_test_raw_enc ], axis=1).astype(np.float32)

# Derive flat_scaled from flat_raw with the same continuous_mask as the main pipeline
k_flat_mean = np.zeros(n_flat, dtype=np.float32)
k_flat_std  = np.ones( n_flat, dtype=np.float32)
k_flat_mean[continuous_mask] = k_flat_raw_train[:, continuous_mask].mean(axis=0).astype(np.float32)
k_flat_std [continuous_mask] = k_flat_raw_train[:, continuous_mask].std(axis=0).astype(np.float32)
k_flat_std  = np.where(k_flat_std == 0, 1.0, k_flat_std)

k_flat_scaled_train = ((k_flat_raw_train - k_flat_mean) / k_flat_std).astype(np.float32)
k_flat_scaled_val   = ((k_flat_raw_val   - k_flat_mean) / k_flat_std).astype(np.float32)
k_flat_scaled_test  = ((k_flat_raw_test  - k_flat_mean) / k_flat_std).astype(np.float32)

# --- Save all k-sigma outputs ---
for split, arrays in {
    'train': (kX_train_norm, S_train_norm, S_train_raw_enc,
              kmask_train, kdelta_train, y_train, train_ids),
    'val':   (kX_val_norm,   S_val_norm,   S_val_raw_enc,
              kmask_val,   kdelta_val,   y_val,   val_ids),
    'test':  (kX_test_norm,  S_test_norm,  S_test_raw_enc,
              kmask_test,  kdelta_test,  y_test,  test_ids),
}.items():
    X, S, S_raw, M, D, y, ids = arrays
    np.save(KSIGMA_DIR / f'X_ts_{split}.npy',             X)
    np.save(KSIGMA_DIR / f'X_static_{split}.npy',         S)
    np.save(KSIGMA_DIR / f'X_static_raw_enc_{split}.npy', S_raw)
    np.save(KSIGMA_DIR / f'mask_{split}.npy',              M)
    np.save(KSIGMA_DIR / f'delta_{split}.npy',             D)
    np.save(KSIGMA_DIR / f'y_{split}.npy',                 y)
    np.save(KSIGMA_DIR / f'ids_{split}.npy',               ids)

for split, (fr, fs) in {
    'train': (k_flat_raw_train, k_flat_scaled_train),
    'val':   (k_flat_raw_val,   k_flat_scaled_val),
    'test':  (k_flat_raw_test,  k_flat_scaled_test),
}.items():
    np.save(KSIGMA_DIR / f'features_flat_raw_{split}.npy', fr)
    np.save(KSIGMA_DIR / f'features_flat_{split}.npy',     fs)

np.save(KSIGMA_DIR / 'ts_norm_mean.npy',       k_norm_mean)
np.save(KSIGMA_DIR / 'ts_norm_std.npy',        k_norm_std)
np.save(KSIGMA_DIR / 'train_impute_means.npy', k_impute_means)
np.save(KSIGMA_DIR / 'flat_norm_mean.npy',     k_flat_mean)
np.save(KSIGMA_DIR / 'flat_norm_std.npy',      k_flat_std)

# Quick sanity checks
assert not np.isnan(kX_train_norm).any(), 'NaN in k-sigma X_train_norm'
assert not np.isnan(k_flat_raw_train).any(), 'NaN in k-sigma flat_raw_train'

# Observation rate comparison
print('Sensitivity analysis: k-sigma processed files saved.')
print(f'  Output directory: {KSIGMA_DIR.resolve()}')
print()
print('Observation rates — Hard bounds vs µ±5σ (train set):')
print(f'  {"Variable":<15s}  {"Hard %":>8s}  {"k5σ %":>8s}  {"Diff":>8s}')
print('  ' + '─' * 44)
for j, col in enumerate(ts_cols):
    r_hard = mask_train[:, :, j].mean() * 100
    r_ksig = kmask_train[:, :, j].mean() * 100
    diff   = r_ksig - r_hard
    flag   = '  ◄' if diff < -2.0 else ''
    print(f'  {col:<15s}  {r_hard:7.1f}%  {r_ksig:7.1f}%  {diff:+7.1f}%{flag}')
print()
print('To run the sensitivity comparison, load from processed/sensitivity_ksigma/')
print('instead of processed/ in your model training notebook.')

Sensitivity analysis: k-sigma processed files saved.
  Output directory: D:\Coding\py\py_Project\DATA5925\processed\sensitivity_ksigma

Observation rates — Hard bounds vs µ±5σ (train set):
  Variable           Hard %     k5σ %      Diff
  ────────────────────────────────────────────
  ALP                  1.6%      1.4%     -0.2%
  ALT                  1.6%      1.1%     -0.5%
  AST                  1.6%      1.1%     -0.6%
  Albumin              1.2%      1.2%     +0.0%
  BUN                  7.2%      7.0%     -0.2%
  Bilirubin            1.6%      1.4%     -0.3%
  Cholesterol          0.2%      0.2%     +0.0%
  Creatinine           7.2%      6.3%     -0.9%
  DiasABP             54.1%     54.3%     +0.2%
  FiO2                16.0%     16.0%     +0.0%
  GCS                 31.8%     31.8%     +0.0%
  Glucose              6.7%      5.8%     -0.9%
  HCO3                 7.0%      7.0%     -0.1%
  HCT                  9.5%      9.5%     +0.0%
  HR                  89.8%     89.8%     +0

---

## Stage 8 — Validation Checks

**21 assertions** that must all pass before the `processed/` directory is used for
model training.

| Check | What is tested |
|---|---|
| V1 | Shape consistency across N, y, ids in each split |
| V2 | X_ts shape: (N, 48, 37) |
| V3 | X_static shape: (N, 10) |
| V4 | ICUType one-hot: each row sums to 1 |
| V5 | Gender column is binary {0, 1} |
| V6 | S_raw_enc and S_norm Age columns differ (z-scoring applied) |
| V7 | FiO2 observed values in [0, 1] after unit conversion |
| V8 | No NaN in imputed/normalised sequence arrays |
| V9 | Mask values in {0, 1} |
| V10 | Delta ≥ 0 |
| V11 | No patient ID overlap between splits |
| V12 | Flat features: no NaN; flat_raw and flat_scaled shapes match |
| V13 | flat_raw and flat_scaled continuous columns differ |
| V14 | Flat feature count = 8 × 37 + 10 = 306 |
| V15 | TroponinI / TroponinT in LOG_TRANSFORM_COLS and PHYS_BOUNDS |
| V16 | flat_scaled non-continuous cols remain in {0, 1} |
| V17 | Irregular split files have no ID overlap |
| V18 | train_impute_means for log vars are in log-transformed scale |
| V19 | X_static shape confirmed: (N, 10) |
| V20 | Height_missing / Weight_missing align with raw missingness |
| V21 | Irregular variable set == ts_cols (37 vars) |

> If any assertion fails, trace back to the relevant stage and re-run from that cell.

In [18]:
# ============================================================
# Validation Checks
# These assertions catch common mistakes. All must pass before
# the processed/ directory is considered ready for model training.
# ============================================================
print('Running validation checks...')

# 1. Shape consistency across splits
assert X_train_norm.shape[0] == len(y_train) == len(train_ids), 'Train size mismatch'
assert X_val_norm.shape[0]   == len(y_val)   == len(val_ids),   'Val size mismatch'
assert X_test_norm.shape[0]  == len(y_test)  == len(test_ids),  'Test size mismatch'
print('  [OK] Shape consistency across N, y, ids')

# 2. Time-series shape
assert X_train_norm.shape[1:] == (N_HOURS, len(ts_cols)), f'X_ts shape wrong: {X_train_norm.shape}'
print(f'  [OK] X_ts shape: (N, {N_HOURS}, {len(ts_cols)})')

# 3. Static feature shape (N, 10): 4 continuous + Gender + 2 missing ind. + 4 ICU one-hot
assert S_train_norm.shape[1]    == 10, f'S_norm shape wrong: {S_train_norm.shape}'
assert S_train_raw_enc.shape[1] == 10, f'S_raw_enc shape wrong: {S_train_raw_enc.shape}'
print('  [OK] X_static shapes: (N, 10) for both S_norm and S_raw_encoded')

# 4. ICUType one-hot: each patient must have exactly one 1 across the 4 ICU columns
icu_row_sums_norm = S_train_norm[:, 6:10].sum(axis=1)
icu_row_sums_raw  = S_train_raw_enc[:, 6:10].sum(axis=1)
assert np.allclose(icu_row_sums_norm, 1.0), 'ICUType one-hot (S_norm) rows do not sum to 1'
assert np.allclose(icu_row_sums_raw,  1.0), 'ICUType one-hot (S_raw_enc) rows do not sum to 1'
print('  [OK] ICUType one-hot: each row sums to 1 in both static versions')

# 5. Gender stays binary {0, 1} in both static versions
assert np.isin(S_train_norm[:, 1].round(6),    [0.0, 1.0]).all(), 'Gender (S_norm) not binary'
assert np.isin(S_train_raw_enc[:, 1].round(6), [0.0, 1.0]).all(), 'Gender (S_raw_enc) not binary'
print('  [OK] Gender column stays in {0, 1} in both static versions')

# 6. S_raw_enc continuous columns differ from S_norm (sanity: raw Age != z-scored Age)
assert not np.allclose(S_train_raw_enc[:, 0], S_train_norm[:, 0]),     'S_raw_enc Age == S_norm Age — z-scoring may not have been applied correctly'
print('  [OK] S_raw_enc and S_norm Age columns are different (expected)')

# 7. FiO2 bounds check on observed values
if 'FiO2' in ts_cols:
    j_fio2 = ts_cols.index('FiO2')
    fio2_obs = X_train_pre_locf[:, :, j_fio2][mask_train[:, :, j_fio2] == 1]
    if len(fio2_obs) > 0:
        assert fio2_obs.max() <= 1.0, f'FiO2 > 1.0 in pre-LOCF data: {fio2_obs.max():.3f}'
        assert fio2_obs.min() >= 0.0, f'FiO2 < 0.0 in pre-LOCF data: {fio2_obs.min():.3f}'
print('  [OK] FiO2 values in [0, 1] after unit conversion')

# 8. No NaN in imputed / normalised sequence arrays
for name, arr in [('X_train_norm', X_train_norm), ('X_val_norm', X_val_norm),
                   ('X_test_norm',  X_test_norm),  ('S_train_norm', S_train_norm)]:
    assert not np.isnan(arr).any(), f'NaN found in {name}'
print('  [OK] No NaN in any imputed/normalised sequence array')

# 9. Mask values are binary
assert np.isin(mask_train, [0.0, 1.0]).all(), 'mask_train contains non-binary values'
print('  [OK] Mask values in {0, 1}')

# 10. Delta is non-negative
assert (delta_train >= 0).all(), 'Negative delta values found'
print('  [OK] Delta >= 0')

# 11. No patient ID overlap between splits
assert len(set(train_ids) & set(val_ids))  == 0, 'LEAKAGE: train/val patient overlap!'
assert len(set(train_ids) & set(test_ids)) == 0, 'LEAKAGE: train/test patient overlap!'
assert len(set(val_ids)   & set(test_ids)) == 0, 'LEAKAGE: val/test patient overlap!'
print('  [OK] No patient ID overlap between splits (no data leakage)')

# 12. Flat features: no NaN, consistent shapes
for tag, fr, fs in [('train', flat_raw_train, flat_scaled_train),
                    ('val',   flat_raw_val,   flat_scaled_val  ),
                    ('test',  flat_raw_test,  flat_scaled_test )]:
    assert not np.isnan(fr).any(), f'NaN in flat_raw_{tag}'
    assert not np.isnan(fs).any(), f'NaN in flat_scaled_{tag}'
    assert fr.shape == fs.shape,   f'flat_raw and flat_scaled shape mismatch ({tag})'
print(f'  [OK] Flat features: no NaN; flat_raw and flat_scaled shapes match ({flat_raw_train.shape})')

# 13. flat_raw and flat_scaled continuous columns differ (sanity: z-score was applied)
assert not np.allclose(flat_raw_train[:, 0], flat_scaled_train[:, 0]),     'flat_raw col 0 == flat_scaled col 0 — z-scoring may not have been applied'
print('  [OK] flat_raw and flat_scaled continuous columns differ (expected)')


# 14. Flat feature count matches design (8 stats × V ts vars + 8 static)
expected_n_flat = 8 * len(ts_cols) + len(STATIC_FINAL_COLS)
assert flat_raw_train.shape[1] == expected_n_flat, \
    f'flat_raw feature count {flat_raw_train.shape[1]} != expected {expected_n_flat}'
assert flat_scaled_train.shape[1] == expected_n_flat, \
    f'flat_scaled feature count {flat_scaled_train.shape[1]} != expected {expected_n_flat}'
print(f'  [OK] Flat feature count: {flat_raw_train.shape[1]} '
      f'= 8 stats × {len(ts_cols)} ts vars + {len(STATIC_FINAL_COLS)} static')


# V15: TroponinI / TroponinT must appear in both LOG_TRANSFORM_COLS and PHYS_BOUNDS
assert 'TroponinI' in LOG_TRANSFORM_COLS, 'TroponinI missing from LOG_TRANSFORM_COLS'
assert 'TroponinT' in LOG_TRANSFORM_COLS, 'TroponinT missing from LOG_TRANSFORM_COLS'
assert 'TroponinI' in PHYS_BOUNDS, 'TroponinI missing from PHYS_BOUNDS'
assert 'TroponinT' in PHYS_BOUNDS, 'TroponinT missing from PHYS_BOUNDS'
print('  [OK] TroponinI / TroponinT present in LOG_TRANSFORM_COLS and PHYS_BOUNDS')

# V16: flat_scaled — truly binary static columns must remain in {0, 1}
#      count (v*8+6) and missing_rate (v*8+7) are NOT z-scored but are also
#      NOT binary, so only check the static-block binary columns.
_binary_cols_flat = [
    n_ts_feats + 1,                              # Gender
    n_ts_feats + 4, n_ts_feats + 5,              # Height_missing, Weight_missing
    n_ts_feats + 6, n_ts_feats + 7,              # ICUType_1, ICUType_2
    n_ts_feats + 8, n_ts_feats + 9,              # ICUType_3, ICUType_4
]
for _i in _binary_cols_flat:
    _vals = np.unique(flat_scaled_train[:, _i].round(6))
    assert set(_vals).issubset({0.0, 1.0}), \
        f'flat_scaled col {_i} should be binary/0-1, got unique vals: {_vals}'
print(f'  [OK] flat_scaled binary cols ({len(_binary_cols_flat)}) all remain in {{0, 1}}')

# V17: irregular split files have no ID overlap
_irr_tr = set(pd.read_parquet(OUTPUT_DIR / 'ts_irregular_train.parquet')['RecordID'])
_irr_va = set(pd.read_parquet(OUTPUT_DIR / 'ts_irregular_val.parquet')['RecordID'])
_irr_te = set(pd.read_parquet(OUTPUT_DIR / 'ts_irregular_test.parquet')['RecordID'])
assert _irr_tr.isdisjoint(_irr_va), 'LEAKAGE: irregular train/val overlap!'
assert _irr_tr.isdisjoint(_irr_te), 'LEAKAGE: irregular train/test overlap!'
assert _irr_va.isdisjoint(_irr_te), 'LEAKAGE: irregular val/test overlap!'
print('  [OK] Irregular split files: no patient ID overlap')

# V18: train_impute_means for log vars should be in log-transformed scale
#      (must be ≤ log1p(raw_obs_mean) by Jensen's inequality for concave log)
for _j in log_col_indices:
    _raw_obs = X_train_pre_locf[:, :, _j][mask_train[:, :, _j] == 1]
    _raw_obs = _raw_obs[~np.isnan(_raw_obs)]
    if len(_raw_obs) > 0:
        _upper = np.log1p(float(_raw_obs.mean())) + 1e-4
        assert float(train_impute_means[_j]) <= _upper, \
            (f'train_impute_means[{ts_cols[_j]}] = {train_impute_means[_j]:.4f} '
             f'appears raw-scale, not log-scale (upper bound: {_upper:.4f})')
print('  [OK] train_impute_means for log vars are in log-transformed scale')

# V19: X_static shape is (N, 10)
assert S_train_norm.shape[1] == 10, f'Expected 10 static features, got {S_train_norm.shape[1]}'
print('  [OK] X_static shape confirmed: (N, 10)')

# V20: Height_missing / Weight_missing align with original raw missingness
#      (static index 4 = Height_missing, index 5 = Weight_missing)
for _df, _sarr, _tag in [(S_train_df, S_train_raw_enc, 'train'),
                          (S_val_df,   S_val_raw_enc,   'val')]:
    _h_exp = _df['Height'].isna().values.astype(float)
    _w_exp = _df['Weight'].isna().values.astype(float)
    assert np.allclose(_sarr[:, 4], _h_exp), f'Height_missing mismatch in {_tag}'
    assert np.allclose(_sarr[:, 5], _w_exp), f'Weight_missing mismatch in {_tag}'
print('  [OK] Height_missing / Weight_missing match original raw missingness')

# V21: irregular variable set matches hourly ts_cols (37 vars each)
_irr_vars = set(pd.read_parquet(OUTPUT_DIR / 'ts_irregular_train.parquet')['Parameter'].unique())
assert _irr_vars == set(ts_cols), \
    f'Irregular/hourly variable mismatch: {_irr_vars.symmetric_difference(set(ts_cols))}'
print(f'  [OK] Irregular variable set matches ts_cols ({len(_irr_vars)} vars)')

print()
print('All validation checks passed. The processed/ directory is ready.')

Running validation checks...
  [OK] Shape consistency across N, y, ids
  [OK] X_ts shape: (N, 48, 37)
  [OK] X_static shapes: (N, 10) for both S_norm and S_raw_encoded
  [OK] ICUType one-hot: each row sums to 1 in both static versions
  [OK] Gender column stays in {0, 1} in both static versions
  [OK] S_raw_enc and S_norm Age columns are different (expected)
  [OK] FiO2 values in [0, 1] after unit conversion
  [OK] No NaN in any imputed/normalised sequence array
  [OK] Mask values in {0, 1}
  [OK] Delta >= 0
  [OK] No patient ID overlap between splits (no data leakage)
  [OK] Flat features: no NaN; flat_raw and flat_scaled shapes match ((3200, 306))
  [OK] flat_raw and flat_scaled continuous columns differ (expected)
  [OK] Flat feature count: 306 = 8 stats × 37 ts vars + 10 static
  [OK] TroponinI / TroponinT present in LOG_TRANSFORM_COLS and PHYS_BOUNDS
  [OK] flat_scaled binary cols (7) all remain in {0, 1}
  [OK] Irregular split files: no patient ID overlap
  [OK] train_impute_mean

---

## Stage 9 — Save All Outputs + Metadata

Write everything to `processed/`.  After this cell completes, the directory is
the single source of truth for all downstream model notebooks.

**Recommendation:** load dimensions and feature names from `metadata.json` rather
than hardcoding `306`, `10`, `37` etc. in model scripts — this keeps model code
robust to future pipeline changes.

```python
import json
meta = json.load(open("processed/metadata.json"))
n_flat    = meta['n_flat_features']   # 306
n_static  = meta['n_static_vars']     # 10
feat_names = meta['flat_feature_names']  # list of 306 names
```

In [19]:
splits_seq = {
    'train': (X_train_norm, S_train_norm, S_train_raw_enc,
              mask_train, delta_train, y_train, train_ids),
    'val'  : (X_val_norm,   S_val_norm,   S_val_raw_enc,
              mask_val,   delta_val,   y_val,   val_ids  ),
    'test' : (X_test_norm,  S_test_norm,  S_test_raw_enc,
              mask_test,  delta_test,  y_test,  test_ids ),
}
splits_flat = {
    'train': (flat_raw_train, flat_scaled_train),
    'val'  : (flat_raw_val,   flat_scaled_val  ),
    'test' : (flat_raw_test,  flat_scaled_test ),
}

for split, (X, S, S_raw, M, D, y, ids) in splits_seq.items():
    np.save(OUTPUT_DIR / f'X_ts_{split}.npy',             X)
    np.save(OUTPUT_DIR / f'X_static_{split}.npy',         S)
    np.save(OUTPUT_DIR / f'X_static_raw_enc_{split}.npy', S_raw)
    np.save(OUTPUT_DIR / f'mask_{split}.npy',             M)
    np.save(OUTPUT_DIR / f'delta_{split}.npy',            D)
    np.save(OUTPUT_DIR / f'y_{split}.npy',                y)
    np.save(OUTPUT_DIR / f'ids_{split}.npy',              ids)

for split, (flat_raw, flat_scaled) in splits_flat.items():
    np.save(OUTPUT_DIR / f'features_flat_raw_{split}.npy', flat_raw)
    np.save(OUTPUT_DIR / f'features_flat_{split}.npy',     flat_scaled)

# Time-series normalisation stats (Track B)
np.save(OUTPUT_DIR / 'ts_norm_mean.npy',       norm_mean)
np.save(OUTPUT_DIR / 'ts_norm_std.npy',        norm_std)
np.save(OUTPUT_DIR / 'train_impute_means.npy', train_impute_means)

# Static normalisation stats
with open(OUTPUT_DIR / 'static_norm_stats.json', 'w') as f:
    json.dump(static_stats, f, indent=2)

# Flat feature normalisation stats (for flat_scaled; fit on train only)
np.save(OUTPUT_DIR / 'flat_norm_mean.npy', flat_mean)
np.save(OUTPUT_DIR / 'flat_norm_std.npy',  flat_std)

# Metadata
flat_feature_names = (
    [f'{col}__{stat}'
     for col in ts_cols
     for stat in ['mean', 'std', 'min', 'max', 'first', 'last', 'count', 'missing_rate']]
    + STATIC_FINAL_COLS
)
meta = {
    'ts_cols':            ts_cols,
    'ts_binary_cols':     TS_BINARY_COLS,
    'static_final_cols':  STATIC_FINAL_COLS,
    'n_hours':            N_HOURS,
    'n_ts_vars':          len(ts_cols),
    'n_static_vars':      len(STATIC_FINAL_COLS),
    'n_patients':         {s: int(len(ids))
                           for s, (_, _, _, _, _, _, ids) in splits_seq.items()},
    'mortality_rate':     {s: float(y.mean())
                           for s, (_, _, _, _, _, y, _) in splits_seq.items()},
    'flat_feature_names': flat_feature_names,
    'n_flat_features':    len(flat_feature_names),
    'log_transform_cols': sorted(LOG_TRANSFORM_COLS),
    'flat_raw_description':    ('8 stats x 37 ts vars + 10 static (original units, '
                                'incl. Height_missing / Weight_missing). '
                                'For XGBoost / RF / LightGBM.'),
    'flat_scaled_description': ('Same 306 features; continuous columns z-scored only. '
                                'Binary/one-hot/missing indicators kept as 0/1. '
                                'For LR / SVM / MLP.'),
}
with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'All outputs saved to: {OUTPUT_DIR.resolve()}')
print()
print(f'  {"File":<50s}  {"Size (KB)":>10s}')
print('  ' + '-' * 63)
for fp in sorted(OUTPUT_DIR.glob('*')):
    size_kb = fp.stat().st_size / 1024
    print(f'  {fp.name:<50s}  {size_kb:>10.1f}')

All outputs saved to: D:\Coding\py\py_Project\DATA5925\processed

  File                                                 Size (KB)
  ---------------------------------------------------------------
  delta_test.npy                                          2775.1
  delta_train.npy                                        22200.1
  delta_val.npy                                           2775.1
  features_flat_raw_test.npy                               478.2
  features_flat_raw_train.npy                             3825.1
  features_flat_raw_val.npy                                478.2
  features_flat_test.npy                                   478.2
  features_flat_train.npy                                 3825.1
  features_flat_val.npy                                    478.2
  flat_norm_mean.npy                                         1.3
  flat_norm_std.npy                                          1.3
  ids_test.npy                                               3.2
  ids_train.npy        

Stage 10 - Machine Learning Part (train-test-valuation)

In [21]:
def prepare_data_cv(train_ids, val_ids, test_ids, df_ts, df_static, y_all, all_record_ids, ts_cols):
    """
    Leakage-free preprocessing for Cross-Validation.
    Uses strictly local variables to prevent interference from global scope.
    """
    
    # 1) Label Extraction
    id_to_y = dict(zip(all_record_ids, y_all))
    y_train = np.array([id_to_y[i] for i in train_ids])
    y_val   = np.array([id_to_y[i] for i in val_ids])
    y_test  = np.array([id_to_y[i] for i in test_ids])

    # 2) Reshape & Apply Physiological Bounds
    X_tr_raw = apply_phys_bounds(df_to_3d(df_ts, train_ids, ts_cols), ts_cols, verbose=False)
    X_va_raw = apply_phys_bounds(df_to_3d(df_ts, val_ids,   ts_cols), ts_cols, verbose=False)
    X_te_raw = apply_phys_bounds(df_to_3d(df_ts, test_ids,  ts_cols), ts_cols, verbose=False)

    # 3) Missingness Mask & Delta
    m_tr = (~np.isnan(X_tr_raw)).astype(np.float32)
    m_va = (~np.isnan(X_va_raw)).astype(np.float32)
    m_te = (~np.isnan(X_te_raw)).astype(np.float32)
    
    delta_tr = compute_delta(m_tr)
    delta_va = compute_delta(m_va)
    delta_te = compute_delta(m_te)

    # 4) Imputation & Transformation
    log_indices = [j for j, col in enumerate(ts_cols) if col in LOG_TRANSFORM_COLS]
    X_tr = apply_log1p_transform(locf_3d(X_tr_raw), log_indices)
    X_va = apply_log1p_transform(locf_3d(X_va_raw), log_indices)
    X_te = apply_log1p_transform(locf_3d(X_te_raw), log_indices)

    # Compute fallback means on TRAIN ONLY
    fb_means = np.array([
        float(np.nanmean(X_tr[:, :, j][m_tr[:, :, j] == 1]))
        if m_tr[:, :, j].sum() > 0 else 0.0
        for j in range(len(ts_cols))
    ], dtype=np.float32)
    
    if 'MechVent' in ts_cols:
        fb_means[ts_cols.index('MechVent')] = 0.0

    X_tr = fill_with_means(X_tr, fb_means)
    X_va = fill_with_means(X_va, fb_means)
    X_te = fill_with_means(X_te, fb_means)

    # 5) Sequential Z-score (Fit on Train)
    norm_m = np.zeros(len(ts_cols), dtype=np.float32)
    norm_s = np.ones(len(ts_cols), dtype=np.float32)
    for j, col in enumerate(ts_cols):
        if col in TS_BINARY_COLS: continue
        obs = X_tr[:, :, j][m_tr[:, :, j] == 1]
        if len(obs) > 1:
            norm_m[j] = float(obs.mean())
            norm_s[j] = float(obs.std()) if obs.std() > 0 else 1.0

    X_tr_norm = zscore(X_tr, norm_m, norm_s)
    X_va_norm = zscore(X_va, norm_m, norm_s)
    X_te_norm = zscore(X_te, norm_m, norm_s)

    # 6) Static Feature Processing
    def process_static_internal(df, stats):
        h_miss = df['Height'].isna().astype(np.float32).values
        w_miss = df['Weight'].isna().astype(np.float32).values
        df_f = df.copy()
        df_f['Age'] = df_f['Age'].fillna(stats['age_m'])
        df_f['Height'] = df_f['Height'].fillna(stats['h_m'])
        df_f['Weight'] = df_f['Weight'].fillna(stats['w_m'])
        df_f['Gender'] = df_f['Gender'].fillna(stats['g_mode'])
        df_f['ICUType'] = df_f['ICUType'].fillna(stats['icu_mode']).astype(int)
        
        age_z = ((df_f['Age'].values - stats['age_m']) / stats['age_s']).astype(np.float32)
        h_z   = ((df_f['Height'].values - stats['h_m']) / stats['h_s']).astype(np.float32)
        w_z   = ((df_f['Weight'].values - stats['w_m']) / stats['w_s']).astype(np.float32)
        
        icu_oh = pd.get_dummies(df_f['ICUType'], prefix='ICUType').reindex(
            columns=['ICUType_1', 'ICUType_2', 'ICUType_3', 'ICUType_4'], fill_value=0
        ).values.astype(np.float32)
        
        return np.column_stack([age_z, df_f['Gender'].values, h_z, w_z, h_miss, w_miss, icu_oh])

    S_tr_df = get_static_df(df_static, train_ids)
    S_va_df = get_static_df(df_static, val_ids)
    S_te_df = get_static_df(df_static, test_ids)

    s_stats = {
        'age_m': S_tr_df['Age'].mean(), 'age_s': max(S_tr_df['Age'].std(), 1e-8),
        'h_m':   S_tr_df['Height'].mean(), 'h_s':   max(S_tr_df['Height'].std(), 1e-8),
        'w_m':   S_tr_df['Weight'].mean(), 'w_s':   max(S_tr_df['Weight'].std(), 1e-8),
        'g_mode': S_tr_df['Gender'].mode().iloc[0] if not S_tr_df['Gender'].dropna().empty else 0.0,
        'icu_mode': int(S_tr_df['ICUType'].mode().iloc[0]) if not S_tr_df['ICUType'].dropna().empty else 1
    }

    S_tr_norm = process_static_internal(S_tr_df, s_stats)
    S_va_norm = process_static_internal(S_va_df, s_stats) # Correctly using local val subset
    S_te_norm = process_static_internal(S_te_df, s_stats)

    # 7) Flat Feature Extraction
    ts_stats_tr = extract_ts_stats(X_tr_raw, m_tr, fb_means, log_indices)
    ts_stats_va = extract_ts_stats(X_va_raw, m_va, fb_means, log_indices)
    ts_stats_te = extract_ts_stats(X_te_raw, m_te, fb_means, log_indices)

    flat_raw_tr = np.concatenate([ts_stats_tr, S_tr_norm], axis=1).astype(np.float32)
    flat_raw_va = np.concatenate([ts_stats_va, S_va_norm], axis=1).astype(np.float32)
    flat_raw_te = np.concatenate([ts_stats_te, S_te_norm], axis=1).astype(np.float32)

    f_m = flat_raw_tr.mean(axis=0)
    f_s = np.where(flat_raw_tr.std(axis=0) == 0, 1.0, flat_raw_tr.std(axis=0))

    flat_sc_tr = (flat_raw_tr - f_m) / f_s
    flat_sc_va = (flat_raw_va - f_m) / f_s
    flat_sc_te = (flat_raw_te - f_m) / f_s

    return {
        "X_train": X_tr_norm, "X_val": X_va_norm, "X_test": X_te_norm,
        "S_train": S_tr_norm, "S_val": S_va_norm, "S_test": S_te_norm,
        "flat_train": flat_sc_tr, "flat_val": flat_sc_va, "flat_test": flat_sc_te,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "mask_train": m_tr, "mask_val": m_va, "mask_test": m_te,
        "delta_train": delta_tr, "delta_val": delta_va, "delta_test": delta_te,
        "train_ids": train_ids, "val_ids": val_ids, "test_ids": test_ids
    }

raw data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ML Model Training & Evaluation — 5-Run Cross Validation
# ML模型训练与评估 — 5次随机划分交叉验证
#
# For each seed, the full preprocessing pipeline is re-executed:
#   - Stratified 80/10/10 split
#   - Static feature imputation (fit on train only)
#   - Flat feature extraction and z-scoring (fit on train only)
# 每次划分都重新执行完整预处理，统计量仅在训练集上拟合，避免数据泄漏。
#
# Class imbalance is handled by random oversampling (upsampling) on training set only,
# consistent with the DL pipeline using sklearn.utils.resample with replace=True.
# 类别不平衡通过随机过采样处理，仅对训练集执行，与DL组保持一致。
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, f1_score,
                             average_precision_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


def find_best_threshold_youden(model, X_val, y_val):
    """
    Select optimal threshold using Youden Index on validation set.
    使用Youden Index在验证集上选择最优决策阈值。
    Youden Index = Sensitivity + Specificity - 1 = TPR + (1 - FPR) - 1
    """
    y_prob               = model.predict_proba(X_val)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    youden_index         = tpr + (1 - fpr) - 1
    return float(thresholds[np.argmax(youden_index)])


def evaluate_model(model, X_test, y_test, threshold):
    """
    Compute all evaluation metrics on test set.
    在测试集上计算所有评估指标。
    """
    y_prob              = model.predict_proba(X_test)[:, 1]
    y_pred              = (y_prob >= threshold).astype(int)
    fpr, tpr, _         = roc_curve(y_test, y_prob)
    tn, fp, fn, tp      = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    prec                = precision_score(y_test, y_pred, zero_division=0)
    rec                 = recall_score(y_test, y_pred, zero_division=0)
    return {
        'AUROC':       auc(fpr, tpr),
        'AUPRC':       average_precision_score(y_test, y_prob),
        'F1':          f1_score(y_test, y_pred, zero_division=0),
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'Threshold':   threshold,
        'Score1':      min(prec, rec),
    }


def random_oversample(X, y, seed):
    """
    Random oversampling: upsample minority class (death) to match majority class (alive).
    随机过采样：将少数类（死亡）上采样至与多数类（存活）数量相等。
    Consistent with DL pipeline using sklearn.utils.resample with replace=True.
    与DL组保持一致，使用replace=True的随机重复抽样。
    """
    X_survive = X[y == 0]
    X_death   = X[y == 1]
    y_survive = y[y == 0]
    y_death   = y[y == 1]

    X_death_up, y_death_up = resample(
        X_death, y_death,
        replace      = True,
        n_samples    = len(X_survive),
        random_state = seed,
    )

    X_bal = np.concatenate([X_survive, X_death_up])
    y_bal = np.concatenate([y_survive, y_death_up])

    idx   = np.random.RandomState(seed).permutation(len(y_bal))
    return X_bal[idx], y_bal[idx]


def run_preprocessing_for_seed(seed):
    """
    Re-execute stratified split + flat feature preprocessing for a given seed.
    针对给定seed重新执行分层划分和平坦特征预处理。
    Reuses parsed data already in memory (df_static, df_ts, all_record_ids, y_all).
    复用内存中已解析的数据，仅重新执行划分和统计量拟合步骤。
    """
    # Stratified split / 分层划分
    train_ids_s, temp_ids_s, y_train_s, y_temp_s = train_test_split(
        all_record_ids, y_all,
        test_size    = VAL_RATIO + TEST_RATIO,
        stratify     = y_all,
        random_state = seed,
    )
    val_ids_s, test_ids_s, y_val_s, y_test_s = train_test_split(
        temp_ids_s, y_temp_s,
        test_size    = TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify     = y_temp_s,
        random_state = seed,
    )

    # Reshape to 3D / 转换为3D数组
    X_train_raw_s = df_to_3d(df_ts, train_ids_s, ts_cols)
    X_val_raw_s   = df_to_3d(df_ts, val_ids_s,   ts_cols)
    X_test_raw_s  = df_to_3d(df_ts, test_ids_s,  ts_cols)

    # Apply physiological bounds / 硬生理边界过滤
    X_train_s = apply_phys_bounds(X_train_raw_s, ts_cols, verbose=False)
    X_val_s   = apply_phys_bounds(X_val_raw_s,   ts_cols, verbose=False)
    X_test_s  = apply_phys_bounds(X_test_raw_s,  ts_cols, verbose=False)

    # Missingness mask / 缺失值掩码
    mask_train_s = (~np.isnan(X_train_s)).astype(np.float32)
    mask_val_s   = (~np.isnan(X_val_s  )).astype(np.float32)
    mask_test_s  = (~np.isnan(X_test_s )).astype(np.float32)

    X_train_pre_s = X_train_s.copy()
    X_val_pre_s   = X_val_s.copy()
    X_test_pre_s  = X_test_s.copy()

    # Static features (fit on train only) / 静态特征（仅在训练集上拟合）
    S_train_df_s  = get_static_df(df_static, train_ids_s)
    S_val_df_s    = get_static_df(df_static, val_ids_s)
    S_test_df_s   = get_static_df(df_static, test_ids_s)

    S_train_raw_s = process_static_raw_encoded(S_train_df_s)
    S_val_raw_s   = process_static_raw_encoded(S_val_df_s)
    S_test_raw_s  = process_static_raw_encoded(S_test_df_s)

    # Flat feature extraction / 平坦特征提取
    fb_means_s = np.zeros(len(ts_cols), dtype=np.float32)
    for j in range(len(ts_cols)):
        obs = X_train_pre_s[:, :, j][mask_train_s[:, :, j] == 1]
        if len(obs) > 0:
            if j in log_col_indices:
                fb_means_s[j] = float(np.nanmean(np.log1p(np.maximum(obs, 0.0))))
            else:
                fb_means_s[j] = float(np.nanmean(obs))
    fb_means_s = np.nan_to_num(fb_means_s, nan=0.0)

    ts_stats_train_s = extract_ts_stats(X_train_pre_s, mask_train_s, fb_means_s, log_col_indices)
    ts_stats_val_s   = extract_ts_stats(X_val_pre_s,   mask_val_s,   fb_means_s, log_col_indices)
    ts_stats_test_s  = extract_ts_stats(X_test_pre_s,  mask_test_s,  fb_means_s, log_col_indices)

    flat_raw_train_s = np.concatenate([ts_stats_train_s, S_train_raw_s], axis=1).astype(np.float32)
    flat_raw_val_s   = np.concatenate([ts_stats_val_s,   S_val_raw_s  ], axis=1).astype(np.float32)
    flat_raw_test_s  = np.concatenate([ts_stats_test_s,  S_test_raw_s ], axis=1).astype(np.float32)

    # Z-score scaling (fit on train only) / 标准化（仅在训练集上拟合）
    flat_mean_s = np.zeros(n_flat, dtype=np.float32)
    flat_std_s  = np.ones( n_flat, dtype=np.float32)
    flat_mean_s[continuous_mask] = flat_raw_train_s[:, continuous_mask].mean(axis=0)
    flat_std_s [continuous_mask] = flat_raw_train_s[:, continuous_mask].std(axis=0)
    flat_std_s  = np.where(flat_std_s == 0, 1.0, flat_std_s)

    flat_scaled_train_s = ((flat_raw_train_s - flat_mean_s) / flat_std_s).astype(np.float32)
    flat_scaled_val_s   = ((flat_raw_val_s   - flat_mean_s) / flat_std_s).astype(np.float32)
    flat_scaled_test_s  = ((flat_raw_test_s  - flat_mean_s) / flat_std_s).astype(np.float32)

    return (flat_scaled_train_s, flat_scaled_val_s, flat_scaled_test_s,
            y_train_s, y_val_s, y_test_s)


# ── Storage for results / 存储结果 ────────────────────────
metric_keys = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity',
               'Threshold', 'Score1']
model_names = ['log_raw', 'svm_raw']
results     = {m: {k: [] for k in metric_keys} for m in model_names}

# ── 5-Fold Cross Validation / 5折交叉验证 ──────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(all_record_ids, y_all)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/5")
    print(f"{'='*60}")

    # Split IDs / 划分ID
    test_ids_fold = all_record_ids[test_idx]
    tv_ids        = all_record_ids[train_val_idx]
    tv_y          = y_all[train_val_idx]

    # Split 15% from train+val as validation set for Youden threshold
    # 从训练集切15%作验证集，用于Youden阈值选择
    tr_ids, vl_ids, _, _ = train_test_split(
        tv_ids, tv_y, test_size=0.15, stratify=tv_y, random_state=fold
    )

    # Re-run preprocessing for this fold / 重新执行预处理
    data_fold = prepare_data_cv(
        train_ids=tr_ids, val_ids=vl_ids, test_ids=test_ids_fold,
        df_ts=df_ts, df_static=df_static, y_all=y_all,
        all_record_ids=all_record_ids, ts_cols=ts_cols
    )

    X_tr = data_fold['flat_train']
    X_va = data_fold['flat_val']
    X_te = data_fold['flat_test']
    y_tr = data_fold['y_train']
    y_va = data_fold['y_val']
    y_te = data_fold['y_test']

    print(f"  Train: {len(y_tr)}  Val: {len(y_va)}  Test: {len(y_te)}")

    # Raw Data
    X_bal, y_bal = X_tr, y_tr
    print(f"{len(y_bal)} samples "
          f"({y_bal.sum():.0f} dead, {(y_bal == 0).sum():.0f} alive)")

    # ── Logistic Regression ────────────────────────────────
    m = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_bal, y_bal)
    thresh  = find_best_threshold_youden(m, X_va, y_va)
    metrics = evaluate_model(m, X_te, y_te, thresh)
    for k in metric_keys:
        results['log_raw'][k].append(metrics[k])
    print(f"  Log  raw — AUROC: {metrics['AUROC']:.4f}  "
          f"F1: {metrics['F1']:.4f}  Threshold: {metrics['Threshold']:.4f}")

    # ── SVM ───────────────────────────────────────────────
    m = SVC(kernel='linear', probability=True, random_state=42)
    m.fit(X_bal, y_bal)
    thresh  = find_best_threshold_youden(m, X_va, y_va)
    metrics = evaluate_model(m, X_te, y_te, thresh)
    for k in metric_keys:
        results['svm_raw'][k].append(metrics[k])
    print(f"  SVM  raw — AUROC: {metrics['AUROC']:.4f}  "
          f"F1: {metrics['F1']:.4f}  Threshold: {metrics['Threshold']:.4f}")

# ── Summary / 汇总 ────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Summary across 5 runs (mean ± std)")
print(f"{'='*60}")

summary_rows = []
for model_name in model_names:
    print(f"\n{model_name}:")
    row = {'Model': model_name}
    for k in metric_keys:
        mean = np.mean(results[model_name][k])
        std  = np.std(results[model_name][k])
        print(f"  {k:<12s}: {mean:.4f} (±{std:.4f})")
        row[f'{k}_mean'] = round(mean, 4)
        row[f'{k}_std']  = round(std, 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'ml_model_comparison_5runs_raw.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR / 'ml_model_comparison_5runs_raw.csv'}")


Fold 1/5
  Train: 2720  Val: 480  Test: 800
2720 samples (377 dead, 2343 alive)
  Log  raw — AUROC: 0.8120  F1: 0.3958  Threshold: 0.0509
  SVM  raw — AUROC: 0.7995  F1: 0.3728  Threshold: 0.1029

Fold 2/5
  Train: 2720  Val: 480  Test: 800
2720 samples (377 dead, 2343 alive)
  Log  raw — AUROC: 0.8518  F1: 0.4048  Threshold: 0.0295
  SVM  raw — AUROC: 0.8447  F1: 0.4824  Threshold: 0.1506

Fold 3/5
  Train: 2720  Val: 480  Test: 800
2720 samples (377 dead, 2343 alive)
  Log  raw — AUROC: 0.8520  F1: 0.4186  Threshold: 0.0421
  SVM  raw — AUROC: 0.8409  F1: 0.4035  Threshold: 0.1120

Fold 4/5
  Train: 2720  Val: 480  Test: 800
2720 samples (377 dead, 2343 alive)
  Log  raw — AUROC: 0.8184  F1: 0.4426  Threshold: 0.0951
  SVM  raw — AUROC: 0.7994  F1: 0.4431  Threshold: 0.1654

Fold 5/5
  Train: 2720  Val: 480  Test: 800
2720 samples (377 dead, 2343 alive)
  Log  raw — AUROC: 0.8396  F1: 0.4749  Threshold: 0.1387
  SVM  raw — AUROC: 0.8236  F1: 0.4452  Threshold: 0.2041

Summary across

upsample

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ML Model Training & Evaluation — 5-Run Cross Validation
# ML模型训练与评估 — 5次随机划分交叉验证
#
# This cell must be run AFTER all preprocessing stages are complete.
# 本cell必须在所有预处理阶段完成后运行。
#
# For each seed, the full preprocessing pipeline is re-executed:
#   - Stratified 80/10/10 split
#   - Static feature imputation (fit on train only)
#   - Flat feature extraction and z-scoring (fit on train only)
# 每次划分都重新执行完整预处理，统计量仅在训练集上拟合，避免数据泄漏。
#
# Class imbalance is handled by random oversampling (upsampling) on training set only,
# consistent with the DL pipeline using sklearn.utils.resample with replace=True.
# 类别不平衡通过随机过采样处理，仅对训练集执行，与DL组保持一致。
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, f1_score,
                             average_precision_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


def find_best_threshold_youden(model, X_val, y_val):
    """
    Select optimal threshold using Youden Index on validation set.
    使用Youden Index在验证集上选择最优决策阈值。
    Youden Index = Sensitivity + Specificity - 1 = TPR + (1 - FPR) - 1
    """
    y_prob               = model.predict_proba(X_val)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    youden_index         = tpr + (1 - fpr) - 1
    return float(thresholds[np.argmax(youden_index)])


def evaluate_model(model, X_test, y_test, threshold):
    """
    Compute all evaluation metrics on test set.
    在测试集上计算所有评估指标。
    """
    y_prob              = model.predict_proba(X_test)[:, 1]
    y_pred              = (y_prob >= threshold).astype(int)
    fpr, tpr, _         = roc_curve(y_test, y_prob)
    tn, fp, fn, tp      = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    prec                = precision_score(y_test, y_pred, zero_division=0)
    rec                 = recall_score(y_test, y_pred, zero_division=0)
    return {
        'AUROC':       auc(fpr, tpr),
        'AUPRC':       average_precision_score(y_test, y_prob),
        'F1':          f1_score(y_test, y_pred, zero_division=0),
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'Threshold':   threshold,
        'Score1':      min(prec, rec),
    }


def random_oversample(X, y, seed):
    """
    Random oversampling: upsample minority class (death) to match majority class (alive).
    随机过采样：将少数类（死亡）上采样至与多数类（存活）数量相等。
    Consistent with DL pipeline using sklearn.utils.resample with replace=True.
    与DL组保持一致，使用replace=True的随机重复抽样。
    """
    X_survive = X[y == 0]
    X_death   = X[y == 1]
    y_survive = y[y == 0]
    y_death   = y[y == 1]

    X_death_up, y_death_up = resample(
        X_death, y_death,
        replace      = True,
        n_samples    = len(X_survive),
        random_state = seed,
    )

    X_bal = np.concatenate([X_survive, X_death_up])
    y_bal = np.concatenate([y_survive, y_death_up])

    idx   = np.random.RandomState(seed).permutation(len(y_bal))
    return X_bal[idx], y_bal[idx]


def run_preprocessing_for_seed(seed):
    """
    Re-execute stratified split + flat feature preprocessing for a given seed.
    针对给定seed重新执行分层划分和平坦特征预处理。
    Reuses parsed data already in memory (df_static, df_ts, all_record_ids, y_all).
    复用内存中已解析的数据，仅重新执行划分和统计量拟合步骤。
    """
    # Stratified split / 分层划分
    train_ids_s, temp_ids_s, y_train_s, y_temp_s = train_test_split(
        all_record_ids, y_all,
        test_size    = VAL_RATIO + TEST_RATIO,
        stratify     = y_all,
        random_state = seed,
    )
    val_ids_s, test_ids_s, y_val_s, y_test_s = train_test_split(
        temp_ids_s, y_temp_s,
        test_size    = TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify     = y_temp_s,
        random_state = seed,
    )

    # Reshape to 3D / 转换为3D数组
    X_train_raw_s = df_to_3d(df_ts, train_ids_s, ts_cols)
    X_val_raw_s   = df_to_3d(df_ts, val_ids_s,   ts_cols)
    X_test_raw_s  = df_to_3d(df_ts, test_ids_s,  ts_cols)

    # Apply physiological bounds / 硬生理边界过滤
    X_train_s = apply_phys_bounds(X_train_raw_s, ts_cols, verbose=False)
    X_val_s   = apply_phys_bounds(X_val_raw_s,   ts_cols, verbose=False)
    X_test_s  = apply_phys_bounds(X_test_raw_s,  ts_cols, verbose=False)

    # Missingness mask / 缺失值掩码
    mask_train_s = (~np.isnan(X_train_s)).astype(np.float32)
    mask_val_s   = (~np.isnan(X_val_s  )).astype(np.float32)
    mask_test_s  = (~np.isnan(X_test_s )).astype(np.float32)

    X_train_pre_s = X_train_s.copy()
    X_val_pre_s   = X_val_s.copy()
    X_test_pre_s  = X_test_s.copy()

    # Static features (fit on train only) / 静态特征（仅在训练集上拟合）
    S_train_df_s  = get_static_df(df_static, train_ids_s)
    S_val_df_s    = get_static_df(df_static, val_ids_s)
    S_test_df_s   = get_static_df(df_static, test_ids_s)

    S_train_raw_s = process_static_raw_encoded(S_train_df_s)
    S_val_raw_s   = process_static_raw_encoded(S_val_df_s)
    S_test_raw_s  = process_static_raw_encoded(S_test_df_s)

    # Flat feature extraction / 平坦特征提取
    fb_means_s = np.zeros(len(ts_cols), dtype=np.float32)
    for j in range(len(ts_cols)):
        obs = X_train_pre_s[:, :, j][mask_train_s[:, :, j] == 1]
        if len(obs) > 0:
            if j in log_col_indices:
                fb_means_s[j] = float(np.nanmean(np.log1p(np.maximum(obs, 0.0))))
            else:
                fb_means_s[j] = float(np.nanmean(obs))
    fb_means_s = np.nan_to_num(fb_means_s, nan=0.0)

    ts_stats_train_s = extract_ts_stats(X_train_pre_s, mask_train_s, fb_means_s, log_col_indices)
    ts_stats_val_s   = extract_ts_stats(X_val_pre_s,   mask_val_s,   fb_means_s, log_col_indices)
    ts_stats_test_s  = extract_ts_stats(X_test_pre_s,  mask_test_s,  fb_means_s, log_col_indices)

    flat_raw_train_s = np.concatenate([ts_stats_train_s, S_train_raw_s], axis=1).astype(np.float32)
    flat_raw_val_s   = np.concatenate([ts_stats_val_s,   S_val_raw_s  ], axis=1).astype(np.float32)
    flat_raw_test_s  = np.concatenate([ts_stats_test_s,  S_test_raw_s ], axis=1).astype(np.float32)

    # Z-score scaling (fit on train only) / 标准化（仅在训练集上拟合）
    flat_mean_s = np.zeros(n_flat, dtype=np.float32)
    flat_std_s  = np.ones( n_flat, dtype=np.float32)
    flat_mean_s[continuous_mask] = flat_raw_train_s[:, continuous_mask].mean(axis=0)
    flat_std_s [continuous_mask] = flat_raw_train_s[:, continuous_mask].std(axis=0)
    flat_std_s  = np.where(flat_std_s == 0, 1.0, flat_std_s)

    flat_scaled_train_s = ((flat_raw_train_s - flat_mean_s) / flat_std_s).astype(np.float32)
    flat_scaled_val_s   = ((flat_raw_val_s   - flat_mean_s) / flat_std_s).astype(np.float32)
    flat_scaled_test_s  = ((flat_raw_test_s  - flat_mean_s) / flat_std_s).astype(np.float32)

    return (flat_scaled_train_s, flat_scaled_val_s, flat_scaled_test_s,
            y_train_s, y_val_s, y_test_s)


# ── Storage for results / 存储结果 ────────────────────────
metric_keys = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity',
               'Threshold', 'Score1']
model_names = ['log_upsample', 'svm_upsample']
results     = {m: {k: [] for k in metric_keys} for m in model_names}

# ── 5-Fold Cross Validation / 5折交叉验证 ──────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(all_record_ids, y_all)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/5")
    print(f"{'='*60}")

    # Split IDs / 划分ID
    test_ids_fold = all_record_ids[test_idx]
    tv_ids        = all_record_ids[train_val_idx]
    tv_y          = y_all[train_val_idx]

    # Split 15% from train+val as validation set for Youden threshold
    # 从训练集切15%作验证集，用于Youden阈值选择
    tr_ids, vl_ids, _, _ = train_test_split(
        tv_ids, tv_y, test_size=0.15, stratify=tv_y, random_state=fold
    )

    # Re-run preprocessing for this fold / 重新执行预处理
    data_fold = prepare_data_cv(
        train_ids=tr_ids, val_ids=vl_ids, test_ids=test_ids_fold,
        df_ts=df_ts, df_static=df_static, y_all=y_all,
        all_record_ids=all_record_ids, ts_cols=ts_cols
    )

    X_tr = data_fold['flat_train']
    X_va = data_fold['flat_val']
    X_te = data_fold['flat_test']
    y_tr = data_fold['y_train']
    y_va = data_fold['y_val']
    y_te = data_fold['y_test']

    print(f"  Train: {len(y_tr)}  Val: {len(y_va)}  Test: {len(y_te)}")

    # Random oversampling on training set only / 仅对训练集进行随机过采样
    X_bal, y_bal = random_oversample(X_tr, y_tr, seed=fold)
    print(f"  After upsampling: {len(y_bal)} samples "
          f"({y_bal.sum():.0f} dead, {(y_bal == 0).sum():.0f} alive)")

    # ── Logistic Regression ────────────────────────────────
    m = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_bal, y_bal)
    thresh  = find_best_threshold_youden(m, X_va, y_va)
    metrics = evaluate_model(m, X_te, y_te, thresh)
    for k in metric_keys:
        results['log_upsample'][k].append(metrics[k])
    print(f"  Log  Upsample — AUROC: {metrics['AUROC']:.4f}  "
          f"F1: {metrics['F1']:.4f}  Threshold: {metrics['Threshold']:.4f}")

    # ── SVM ───────────────────────────────────────────────
    m = SVC(kernel='linear', probability=True, random_state=42)
    m.fit(X_bal, y_bal)
    thresh  = find_best_threshold_youden(m, X_va, y_va)
    metrics = evaluate_model(m, X_te, y_te, thresh)
    for k in metric_keys:
        results['svm_upsample'][k].append(metrics[k])
    print(f"  SVM  Upsample — AUROC: {metrics['AUROC']:.4f}  "
          f"F1: {metrics['F1']:.4f}  Threshold: {metrics['Threshold']:.4f}")

# ── Summary / 汇总 ────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Summary across 5 runs (mean ± std)")
print(f"{'='*60}")

summary_rows = []
for model_name in model_names:
    print(f"\n{model_name}:")
    row = {'Model': model_name}
    for k in metric_keys:
        mean = np.mean(results[model_name][k])
        std  = np.std(results[model_name][k])
        print(f"  {k:<12s}: {mean:.4f} (±{std:.4f})")
        row[f'{k}_mean'] = round(mean, 4)
        row[f'{k}_std']  = round(std, 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'ml_model_comparison_5runs_upsampling.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR / 'ml_model_comparison_5runs_upsampling.csv'}")


Fold 1/5
  Train: 2720  Val: 480  Test: 800
  After upsampling: 4686 samples (2343 dead, 2343 alive)
  Log  Upsample — AUROC: 0.8089  F1: 0.4035  Threshold: 0.2006
  SVM  Upsample — AUROC: 0.8041  F1: 0.3966  Threshold: 0.2386

Fold 2/5
  Train: 2720  Val: 480  Test: 800
  After upsampling: 4686 samples (2343 dead, 2343 alive)
  Log  Upsample — AUROC: 0.8354  F1: 0.4765  Threshold: 0.4184
  SVM  Upsample — AUROC: 0.8092  F1: 0.4676  Threshold: 0.5293

Fold 3/5
  Train: 2720  Val: 480  Test: 800
  After upsampling: 4686 samples (2343 dead, 2343 alive)
  Log  Upsample — AUROC: 0.8482  F1: 0.4338  Threshold: 0.1939
  SVM  Upsample — AUROC: 0.8373  F1: 0.4000  Threshold: 0.2027

Fold 4/5
  Train: 2720  Val: 480  Test: 800
  After upsampling: 4686 samples (2343 dead, 2343 alive)
  Log  Upsample — AUROC: 0.8089  F1: 0.4710  Threshold: 0.5299
  SVM  Upsample — AUROC: 0.7927  F1: 0.4078  Threshold: 0.2957

Fold 5/5
  Train: 2720  Val: 480  Test: 800
  After upsampling: 4686 samples (2343 dead

SHAP Analysis for Log_upsampling & svm_upsampling seed = 0

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
import joblib

# Train and save models on seed=0 split for SHAP and ICE analysis
# 用seed=42的划分训练并保存模型，用于后续SHAP和ICE分析
SHAP_SEED = 0

(X_tr, X_va, X_te,
 y_tr, y_va, y_te) = run_preprocessing_for_seed(SHAP_SEED)

# Upsample training set / 对训练集进行随机过采样
X_bal, y_bal = random_oversample(X_tr, y_tr, SHAP_SEED)

# Save test set for SHAP/ICE analysis / 保存测试集
np.save('X_test_shap.npy', X_te)
np.save('y_test_shap.npy', y_te)

# Train and save Logistic Regression / 训练并保存逻辑回归模型
model_log = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
model_log.fit(X_bal, y_bal)
joblib.dump(model_log, 'logistic_upsample_model.pkl')
print("Saved: logistic_upsample_model.pkl")

# Train and save SVM / 训练并保存SVM模型
model_svm = SVC(kernel='linear', probability=True, random_state=42)
model_svm.fit(X_bal, y_bal)
joblib.dump(model_svm, 'svm_upsample_model.pkl')
print("Saved: svm_upsample_model.pkl")

print("Saved: X_test_shap.npy, y_test_shap.npy")

Saved: logistic_upsample_model.pkl
Saved: svm_upsample_model.pkl
Saved: X_test_shap.npy, y_test_shap.npy


---

## Appendix — Quick Reference: Loading Data for Your Model

Run this preprocessing notebook **once** to populate `processed/`.
Then copy the relevant block into your model training notebook.

> Load dimension constants from `metadata.json` to stay resilient to pipeline changes:
> ```python
> import json
> meta = json.load(open("processed/metadata.json"))
> n_flat = meta['n_flat_features']   # 306
> ts_cols = meta['ts_cols']          # list of 37 variable names
> ```

---

### For XGBoost / RF / LightGBM  —  Stage 4 flat_raw (unscaled)

```python
import numpy as np

X_train = np.load('processed/features_flat_raw_train.npy')  # (3200, 306)
X_val   = np.load('processed/features_flat_raw_val.npy')    # (400,  306)
X_test  = np.load('processed/features_flat_raw_test.npy')   # (400,  306)
y_train = np.load('processed/y_train.npy')                  # (3200,)
y_val   = np.load('processed/y_val.npy')                    # (400,)
y_test  = np.load('processed/y_test.npy')                   # (400,)
```

Feature names (for SHAP): `processed/metadata.json` → `flat_feature_names` (306 entries)

---

### For Logistic Regression / SVM / MLP  —  Stage 4 flat_scaled (continuous cols z-scored)

```python
import numpy as np

X_train = np.load('processed/features_flat_train.npy')  # (3200, 306)
X_val   = np.load('processed/features_flat_val.npy')    # (400,  306)
X_test  = np.load('processed/features_flat_test.npy')   # (400,  306)
y_train = np.load('processed/y_train.npy')
```

Scaler params (continuous columns only): `flat_norm_mean.npy`, `flat_norm_std.npy` (306-dim;
zeros/ones for binary/indicator columns so those columns are unchanged).

---

### For LSTM / GRU / Transformer  —  Stage 5 sequence (no missingness input)

```python
import numpy as np, torch

X_ts     = torch.tensor(np.load('processed/X_ts_train.npy'))      # (3200, 48, 37)
X_static = torch.tensor(np.load('processed/X_static_train.npy'))  # (3200, 10)
y        = torch.tensor(np.load('processed/y_train.npy'))         # (3200,)
```

Static layout: `[Age_z, Gender, Height_z, Weight_z, Height_missing, Weight_missing, ICUType_1..4]`

**Combining time-series and static:**
- Pattern A (concat at each step): `torch.cat([X_ts, X_static.unsqueeze(1).expand(-1, 48, -1)], dim=2)`
- Pattern B (inject at classifier head — recommended): `torch.cat([rnn_final_state, X_static], dim=1)`

---

### For GRU-D / BiT-MAC (hourly)  —  Stage 5 sequence + missingness signals

```python
import numpy as np, torch

X_ts     = torch.tensor(np.load('processed/X_ts_train.npy'))      # (3200, 48, 37)
X_static = torch.tensor(np.load('processed/X_static_train.npy'))  # (3200, 10)
mask     = torch.tensor(np.load('processed/mask_train.npy'))       # (3200, 48, 37) — 1=observed
delta    = torch.tensor(np.load('processed/delta_train.npy'))      # (3200, 48, 37) — hours since last obs
y        = torch.tensor(np.load('processed/y_train.npy'))          # (3200,)
```

Replace `train` → `val` / `test` for evaluation splits.

---

### For BiT-MAC (irregular timestamps)  —  Stage 6

```python
import pandas as pd

df_train = pd.read_parquet('processed/ts_irregular_train.parquet')
# Columns: RecordID (int), minute (int), Parameter (str), Value (float32)
# No further filtering needed — each file contains only that split's patients.
```

---

### Class imbalance — reminder

Dataset mortality rate ~13.9%.  Use one of:
- `pos_weight = (1 - rate) / rate ≈ 6.2` in `BCEWithLogitsLoss` (PyTorch)
- `scale_pos_weight = 6.2` in XGBoost
- `class_weight = 'balanced'` in scikit-learn
- Threshold optimisation via Youden index on the validation set

Report all evaluation metrics per PhysioNet/CinC 2012 protocol:
**AUROC, AUPRC** (primary), Sensitivity, Specificity, PPV, NPV, F1, Brier score.

---

### Complete output file inventory

| File | Shape | Description |
|---|---|---|
| `features_flat_raw_{split}.npy` | (N, **306**) | Flat features, unscaled |
| `features_flat_{split}.npy` | (N, **306**) | Flat features, continuous cols z-scored |
| `flat_norm_mean/std.npy` | (306,) | Scaler params (0/1 for binary/indicator cols) |
| `X_ts_{split}.npy` | (N, 48, 37) | Sequence: LOCF → log1p → z-scored |
| `X_static_{split}.npy` | (N, **10**) | Static: z-scored continuous + binary indicators |
| `X_static_raw_enc_{split}.npy` | (N, **10**) | Static: original units + binary indicators |
| `mask_{split}.npy` | (N, 48, 37) | 1 = observed, 0 = imputed |
| `delta_{split}.npy` | (N, 48, 37) | Hours since last observation |
| `y_{split}.npy` | (N,) | Binary label (0 = survive, 1 = death) |
| `ids_{split}.npy` | (N,) | RecordID for traceability |
| `ts_norm_mean/std.npy` | (37,) | Z-score params for time-series |
| `train_impute_means.npy` | (37,) | Per-variable fallback means (log-scale for log vars) |
| `ts_irregular_{split}.parquet` | — | Raw long format: RecordID, minute, Parameter, Value |
| `static_norm_stats.json` | — | Age/Height/Weight mean+std, Gender/ICUType mode |
| `metadata.json` | — | ts_cols, flat_feature_names (306), split sizes, mortality rates |